In [ ]:
# Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

# Start - Data Preprocessing

## Data preparation

In [ ]:
import os, re
import numpy as np
import pandas as pd

# ======================================================================
# STEP 1: LOAD AND PREPARE DATA
#   Multiclass label:
#     0 = Healthy (everything else, incl. H and "0")
#     1 = Auxiliary leakage (A,B)
#     2 = Combined leakage (C,D,E,F,G)
# ======================================================================

def load_data(model_path, monorail_paths):

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking and good BC acquisition start
        df = df[df['Non_Standard_Braking'] == 0]
        df = df[df['BC_BadStart'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_Dati06.csv" -> 6
        match = re.search(r'Dati(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source

        # Keep Malfunction column present (Monorail usually unlabeled)
        df['Malfunction'] = 0

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # =========================
    # MULTICLASS LABEL (CHANGE)
    # =========================
    aux_codes      = ['A', 'B']
    leakage_codes  = ['C', 'D', 'E', 'F', 'G']

    df_reference['LeakageLabel'] = np.select(
        [
            df_reference['Malfunction'].isin(aux_codes),
            df_reference['Malfunction'].isin(leakage_codes),
        ],
        [
            'Auxiliary leakage',
            'Combined leakage',
        ],
        default='Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1

    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # ==========================================================
    # ENCODE FINAL LABEL COLUMN (CHANGE: now 0/1/2, Monorail NaN)
    # ==========================================================
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({
            'Healthy': 0,
            'Auxiliary leakage': 1,
            'Combined leakage': 2
        })

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue

    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
        lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
    )

    df_base = df_combined.copy()
    return df_base, df_reference_subset, df_data_subset


In [ ]:
model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_raw_Dati01.csv',
    'TestBrakefinal_data_raw_Dati06.csv',
    'TestBrakefinal_data_raw_Dati27.csv'
]

[df, df_reference, df_monorail] = load_data(model_path, monorail_paths)
df['Malfunction'] = df['Malfunction'].astype(str)
print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail

In [ ]:
df_aux = df[df['label'].eq(1)].copy()
print(df_aux.shape)
df_aux.head()

## PREPROCESS DATA FOR ML

## Data Exploration Continues

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

# ------------------------------------------------------------
# 1. Data prep
# ------------------------------------------------------------
rng = np.random.default_rng(42)

df_plot = df.copy()
df_plot["y_jitter"] = rng.normal(0, 0.02, size=len(df_plot))

feat = "Total_power_efficiency"
df_plot[feat] = pd.to_numeric(df_plot[feat], errors="coerce")

df_filtered = df_plot[df_plot["WV_bin"] == 1].copy()
df_filtered = df_filtered.dropna(subset=[feat])

# ------------------------------------------------------------
# 2. DISTINCT color mapping (robust)
# ------------------------------------------------------------

# ---- Kit Source colors ----
sources = sorted(df_filtered["Source"].unique())
palette_source = sns.color_palette("tab10", n_colors=len(sources))
source_color_map = dict(zip(sources, palette_source))

df_filtered["color_source"] = df_filtered["Source"].map(source_color_map)

# ---- Leakage label colors (explicit & fixed) ----
label_color_map = {
    0: "#1f77b4",  # blue   (healthy)
    1: "#ff7f0e",  # orange (leakage)
    2: "#d62728",  # red    (severe)
}
df_filtered["color_label"] = df_filtered["label"].map(label_color_map)

# ------------------------------------------------------------
# 3. Plot
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: by Source ----------------
axes[0].scatter(
    df_filtered[feat],
    df_filtered["y_jitter"],
    c=df_filtered["color_source"],
    alpha=0.75,
    edgecolors="black",
    linewidths=0.3
)

axes[0].set_title("Total Power Efficiency distribution by Kit Source")
axes[0].set_xlabel(feat)
axes[0].set_yticks([])

# Correct legend (Source)
for s in sources:
    axes[0].scatter(
        [], [], 
        color=source_color_map[s],
        label=f"Source {s}",
        edgecolors="black"
    )
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: by Label ----------------
axes[1].scatter(
    df_filtered[feat],
    df_filtered["y_jitter"],
    c=df_filtered["color_label"],
    alpha=0.75,
    edgecolors="black",
    linewidths=0.3
)

axes[1].set_title("Total Power Efficiency distribution by Label")
axes[1].set_xlabel(feat)
axes[1].set_yticks([])

# Correct legend (Label)
for lab, col in label_color_map.items():
    axes[1].scatter([], [], color=col, label=f"Label {lab}", edgecolors="black")
axes[1].legend(title="Leakage Label", loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
print("Max in df:", pd.to_numeric(df[feat], errors="coerce").max())
print("Max in df_filtered:", pd.to_numeric(df_filtered[feat], errors="coerce").max())

print(df_filtered.loc[pd.to_numeric(df_filtered[feat], errors="coerce") > 13,
                      ["Source", "label", feat]].head(20))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df_plot['y_jitter'] = rng.normal(0, 0.02, size=len(df_plot))
# df_filtered = df_plot[(df_plot['Max_pressure_pipe'] < 0.8) & (df_plot['WV_bin'] == 1)].copy()
# df_filtered = df_plot[(df_plot['EmergencyBrake_action'] == 0) & (df_plot['WV_bin'] == 1)].copy()
df_filtered = df_plot[
    (df_plot['DataSource'] == 0) & 
    (df_plot['WV_bin'].isin([1]))
].copy()
# df_filtered = df_plot[(df_plot['DataSource'] == 0) & (df_plot['WV_bin'] == 1)].copy()
# df_filtered = df_plot[(df_plot['WV_bin'] == 1)].copy()
# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df_plot['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
    2: (0.9, 0.1, 0.1, 1.0),    # solid red
}


# palette_label  = sns.color_palette("Set1", n_colors=len(df_plot['label'].unique()))

# Map categories to colors
source_codes = df_filtered['Source'].astype('category').cat.codes
label_codes  = df_filtered['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Delay distribution by Kit Source")
axes[0].set_xlabel("Total Power Delay")
axes[0].set_yticks([])

# Source legend
sources = df_filtered['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Delay distribution by Label")
axes[1].set_xlabel("Total Power Delay")
axes[1].set_yticks([])

# Label legend
labels = df_filtered['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")
axes[1].set_xlim(-1, 2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    Plot Total_power_efficiency against each selected feature.
    Creates N separate figures (one per feature).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data containing 'Total_power_efficiency', 'label', and selected features.
    features : list of str
        List of feature column names to plot against Total_power_efficiency.
    base_width : int, optional
        Width of each figure (default=7).
    base_height : int, optional
        Height of each figure (default=5).
    x_limits : tuple (min, max), optional
        Limits for the X-axis (Total_power_efficiency).
    """
    # Masks for labels
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1
    mask_2 = df['label'] == 2
    
    for feature in features:
        plt.figure(figsize=(base_width, base_height))
        
        plt.scatter(
            df.loc[mask_0, 'Total_power_efficiency'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='0'
        )
        plt.scatter(
            df.loc[mask_1, 'Total_power_efficiency'],
            df.loc[mask_1, feature],
            color='green', alpha=0.7, label='1'
        )
        
        plt.scatter(
            df.loc[mask_2, 'Total_power_efficiency'],
            df.loc[mask_2, feature],
            color='red', alpha=0.7, label='2'
        )
        
        plt.xlabel("Total_power_efficiency")
        plt.ylabel(feature)
        plt.title(f"Total_power_efficiency vs {feature}")
        plt.legend(title='label', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Apply X-axis limits if provided
        if x_limits is not None:
            plt.xlim(x_limits)
        
        plt.tight_layout()
        plt.show()

# Example usage:
df_filtered = df_plot[df_plot['WV_bin'] == 1]

selected_features = ["Total_power_delay", "WV_MeanPressure","Total_energy_efficiency","Std_delay_exp","Total_energy_delay"]

# Limit X-axis between 0 and 100
plot_efficiency_vs_features(df_filtered, selected_features, x_limits=(0, 10))


In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    For each feature in `features`, create ONE figure with two subplots:

    - LEFT: Total_power_efficiency vs feature, colored by binary df['label'] (0 / 1)
    - RIGHT: Total_power_efficiency vs feature, colored by df['Malfunction'] groups:
        * C, E, G -> green
        * D, F     -> purple
        * 0        -> blue
        * A, B     -> cyan
    """
    # Masks for labels (left plot)
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1
    mask_2 = df['label'] == 2

    # Convenience: Malfunction column (assumed to exist)
    mal = df['Malfunction']

    # Define malfunction groups (right plot)
    group_defs = {
        "Healthy / 0":            (mal == 0) | (mal == '0'),
        "2.5 mm efflux (C,E,G)":   mal.isin(['C', 'E', 'G']),
        "1 mm efflux (D,F)":     mal.isin(['D', 'F']),
        "Auxilarry Leakage (A,B)":   mal.isin(['A', 'B']),
    }

    group_colors = {
        "Healthy / 0":            'blue',
        "2.5 mm efflux (C,E,G)":   'purple',
        "1 mm efflux (D,F)":     'red',
        "Auxilarry Leakage (A,B)":   'green',
    }

    for feature in features:
        fig, axes = plt.subplots(
            1, 2,
            figsize=(2 * base_width, base_height),
            sharex=True,  # same Total_power_efficiency scale
            sharey=False  # y scale may differ feature to feature
        )

        # -------------------------------------------------
        # LEFT: binary label (0/1) as in your original code
        # -------------------------------------------------
        ax_left = axes[0]

        ax_left.scatter(
            df.loc[mask_0, 'Total_power_efficiency'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='label = 0'
        )
        ax_left.scatter(
            df.loc[mask_1, 'Total_power_efficiency'],
            df.loc[mask_1, feature],
            color='green', alpha=0.7, label='label = 1'
        )
        ax_left.scatter(
            df.loc[mask_2, 'Total_power_efficiency'],
            df.loc[mask_2, feature],
            color='red', alpha=0.7, label='label = 2'
        )
        ax_left.set_xlabel("Total_power_efficiency")
        ax_left.set_ylabel(feature)
        # ax_left.set_title(f"Total_power_efficiency vs {feature}")
        ax_left.legend(loc='best')

        if x_limits is not None:
            ax_left.set_xlim(x_limits)

        # -------------------------------------------------
        # RIGHT: malfunction-based coloring
        # -------------------------------------------------
        ax_right = axes[1]

        # plot each group with its own color
        for group_name, group_mask in group_defs.items():
            if group_mask.any():
                ax_right.scatter(
                    df.loc[group_mask, 'Total_power_efficiency'],
                    df.loc[group_mask, feature],
                    color=group_colors[group_name],
                    alpha=0.7,
                    label=group_name
                )

        ax_right.set_xlabel("Total_power_efficiency")
        ax_right.set_ylabel(feature)
        # ax_right.set_title(f"Malfunction groups: Total_power_efficiency vs {feature}", fontsize=10)
        ax_right.legend(loc='upper right', fontsize=12)


        if x_limits is not None:
            ax_right.set_xlim(x_limits)
        fig.suptitle(f"Total_power_efficiency vs {feature}", fontsize=16)
        fig.tight_layout()
        plt.show()


# Example usage:
df_filtered = df_plot[df_plot['WV_bin'] == 1]
plot_features = [
    "Total_power_delay",
    "WV_MeanPressure",
    "Total_energy_efficiency",
    "Std_delay_exp",
    "Mean_delay_exp"
]

plot_efficiency_vs_features(df_filtered, plot_features, x_limits=None)


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

def preprocess_data(df, features, test_size,
                    wv_bins=(1,),
                    label_col="label",
                    mal_col="Malfunction",
                    random_state=42):


    df = df.copy()

    # ---------------------------
    # 1) Regime filtering
    # ---------------------------
    df_filt = df[df["WV_bin"].isin(wv_bins)].copy()

    missing = [f for f in features if f not in df_filt.columns]
    if missing:
        raise ValueError(f"Missing features: {missing}")

    X     = df_filt[features]
    y     = df_filt[label_col]
    y_mal = df_filt[mal_col]

    # ---------------------------
    # 2) Train / test split
    # ---------------------------
    X_train, X_test, y_train, y_test, mal_train, mal_test = train_test_split(
        X, y, y_mal,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # ---------------------------
    # 3) Median imputation (FIT ON TRAIN ONLY)
    # ---------------------------
    imputer = SimpleImputer(strategy="median")

    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test_imp = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    # Healthy-only subset (already imputed)
    X_train_healthy = X_train_imp[y_train == 0]

    # ---------------------------
    # 4) Diagnostics
    # ---------------------------
    print(f"Total samples: {len(df)}")
    print(f"Filtered samples (WV_bin==1): {len(df_filt)}")
    print(f"Training samples: {len(X_train)} "
          f"(Healthy: {(y_train==0).sum()}, Aux Leakage: {(y_train==1).sum()}, Combined Leakage: {(y_train==2).sum()})")
    print(f"Training samples (healthy only): {len(X_train_healthy)}")
    print(f"Test samples: {len(X_test)} "
          f"(Healthy: {(y_test==0).sum()}, Aux Leakage: {(y_test==1).sum()}, Combined Leakage: {(y_test==2).sum()})")

    return (
        X_train_imp, X_test_imp,
        y_train, y_test,
        mal_train, mal_test,
        X_train_healthy,
        imputer   # <-- IMPORTANT: keep it for later inference
    )


# Algorithm Start - Select Features

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING TRAIN TEST SPLIT AND FEATURE SELECTION 
# ============================================================================
selected_features = ['Total_power_efficiency','Std_delay_exp'] 
# selected_features = ["Total_power_efficiency","Total_power_delay","Std_delay_exp","Total_energy_efficiency","Total_energy_delay"]
# selected_features = ["Total_power_efficiency","Std_delay_exp","Total_energy_efficiency"]

[X_train, X_test, y_train, y_test, mal_train, mal_test, X_train_healthy, imputer] = preprocess_data(df, selected_features, test_size=0.2, wv_bins=(1,), random_state=42)
X_train.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import skew, kurtosis, shapiro, normaltest

for feature in selected_features:
    plt.figure(figsize=(6,4))
    sns.histplot(X_train[feature], kde=True, bins=30, color="steelblue")
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.show()
    

# Create a results dictionary
results = {}

for feature in selected_features:
    data = X_train[feature].dropna()  # drop NaNs if any
    
    # Metrics
    skewness = skew(data)
    kurt = kurtosis(data)
    
    results[feature] = {
        "Skewness": skewness,
        "Kurtosis": kurt,
    }

# Convert to DataFrame for readability
metrics_df = pd.DataFrame(results).T
print(metrics_df)

### Downsampling Method to select only Borderline Points

reference: A scalable fuzzy support vector machine for fault detection in
transportation systems
Jie Liu a, Enrico Zio (2018)

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def reverse_nearest_neighbors_outlier_detection(X, y, k=5):
    """
    RNN outlier detection that preserves original indices.

    Returns
    -------
    X_cleaned : same type as X
    y_cleaned : same type as y
    outlier_indices : np.ndarray (original indices of outliers)
    kept_indices : np.ndarray (original indices of kept points)
    """
    X_arr = np.asarray(X)
    y_arr = np.asarray(y)
    n_samples = len(X_arr)

    if hasattr(X, "index"):
        original_idx = X.index.to_numpy()
    else:
        original_idx = np.arange(n_samples)

    classes = np.unique(y_arr)
    reverse_neighbor_count = np.zeros(n_samples)

    for class_label in classes:
        class_indices = np.where(y_arr == class_label)[0]
        X_class = X_arr[class_indices]

        if len(class_indices) <= k:
            reverse_neighbor_count[class_indices] = 1
            continue

        nbrs = NearestNeighbors(n_neighbors=min(k + 1, len(X_class)))
        nbrs.fit(X_class)
        _, indices = nbrs.kneighbors(X_class)

        for i, neighbors in enumerate(indices):
            for neighbor_idx in neighbors[1:]:     # skip self
                actual_idx = class_indices[neighbor_idx]
                reverse_neighbor_count[actual_idx] += 1

    outlier_mask = reverse_neighbor_count == 0
    clean_mask   = ~outlier_mask

    outlier_indices = original_idx[outlier_mask]
    kept_indices    = original_idx[clean_mask]

    if hasattr(X, "iloc"):
        X_cleaned = X.iloc[clean_mask].copy()
    else:
        X_cleaned = X_arr[clean_mask]

    if hasattr(y, "iloc"):
        y_cleaned = y.iloc[clean_mask].copy()
    else:
        y_cleaned = y_arr[clean_mask]

    print(f"Total samples: {n_samples}")
    print(f"Outliers detected: {len(outlier_indices)}")
    print(f"Samples after cleaning: {len(X_cleaned)}")

    return X_cleaned, y_cleaned, outlier_indices, kept_indices


def knn_borderline_selection(X, y, indices=None, k=5, verbose=True):
    """
    Select borderline points using k-Nearest Neighbors, preserving original indices.

    Parameters
    ----------
    X : array-like or DataFrame
    y : array-like or Series
    indices : array-like or None
        Original indices of the rows in X,y. If None, use 0..n-1.
    k : int
        Number of neighbors.

    Returns
    -------
    X_borderline : same type as X
    y_borderline : same type as y
    borderline_indices : np.ndarray (original indices of BORDERLINE points)
    """
    X_arr = np.asarray(X)
    y_arr = np.asarray(y)
    n_samples = len(X_arr)

    if indices is None:
        indices = np.arange(n_samples)
    else:
        indices = np.asarray(indices)

    nbrs = NearestNeighbors(n_neighbors=min(k + 1, n_samples))
    nbrs.fit(X_arr)
    distances, nn_indices = nbrs.kneighbors(X_arr)

    borderline_mask = np.zeros(n_samples, dtype=bool)

    for i in range(n_samples):
        neighbor_idx = nn_indices[i][1:k+1]          # skip self
        neighbor_labels = y_arr[neighbor_idx]
        if len(np.unique(neighbor_labels)) > 1:
            borderline_mask[i] = True

    borderline_indices = indices[borderline_mask]

    if hasattr(X, "iloc"):
        X_borderline = X.iloc[borderline_mask].copy()
    else:
        X_borderline = X_arr[borderline_mask]

    if hasattr(y, "iloc"):
        y_borderline = y.iloc[borderline_mask].copy()
    else:
        y_borderline = y_arr[borderline_mask]

    if verbose:
        print(f"\nTotal samples: {n_samples}")
        print(f"Data reduction: {100 * (1 - len(X_borderline)/n_samples):.1f}% removed")

    return X_borderline, y_borderline, borderline_indices

def complete_preprocessing(X, y, k_outlier=5, k_borderline=5):
    """
    Outlier removal (RNN) + borderline selection (KNN),
    with full index tracking.
    """
    print("Step 1: Outlier Detection using RNN")
    X_clean, y_clean, out_idx, kept_idx = reverse_nearest_neighbors_outlier_detection(
        X, y, k=k_outlier
    )


    print("Step 2: Borderline Point Selection using KNN")
    X_final, y_final, final_idx = knn_borderline_selection(
        X_clean, y_clean, indices=kept_idx, k=k_borderline
    )


    print(f"Data reduction: {len(X)} → {len(X_final)} "
          f"({100 * len(X_final) / len(X):.2f}%)")

    # out_idx: original outliers
    # kept_idx: original non-outliers (after RNN)
    # final_idx: original indices of borderline, non-outlier points
    return X_final, y_final, out_idx, kept_idx, final_idx



In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.stats import median_abs_deviation  # robust scale (MAD)

def statistical_outlier_removal(
    X, y,
    healthy_label=0,
    confidence=0.99,
    method="zscore",          # "zscore" or "mad"
    tail="upper",             # "upper" (only high end), "lower", or "two"
    eps=1e-12,
    verbose=True
):
    """
    Fit thresholds ONLY on healthy data (y==healthy_label) with given confidence,
    then REMOVE only healthy samples that violate the threshold(s).
    Fault samples are never removed.

    tail rule per feature:
      - tail="upper":  x <= high  (remove if x > high)
      - tail="lower":  x >= low   (remove if x < low)
      - tail="two":    low <= x <= high (remove if outside)

    Returns
    -------
    X_cleaned, y_cleaned, removed_indices, kept_indices, thresholds(dict)
    where indices are original indices if X has .index, else 0..n-1.
    """

    X_arr = np.asarray(X)
    y_arr = np.asarray(y)
    n_samples, n_features = X_arr.shape[0], X_arr.shape[1]

    # Preserve original indices
    if hasattr(X, "index"):
        original_idx = X.index.to_numpy()
    else:
        original_idx = np.arange(n_samples)

    # Healthy subset (fit only here)
    healthy_mask = (y_arr == healthy_label)
    healthy_pos = np.where(healthy_mask)[0]

    if len(healthy_pos) < 5:
        if verbose:
            print(f"[WARN] Only {len(healthy_pos)} healthy samples. Skipping removal.")
        removed_indices = np.array([], dtype=original_idx.dtype)
        kept_indices = original_idx.copy()
        if hasattr(X, "iloc"):
            return X.copy(), y.copy(), removed_indices, kept_indices, {}
        return X_arr, y_arr, removed_indices, kept_indices, {}

    Xh = X_arr[healthy_pos]

    # One-sided vs two-sided z
    alpha = 1.0 - float(confidence)
    tail = str(tail).lower()
    if tail == "two":
        z = norm.ppf(1.0 - alpha / 2.0)
    elif tail in ("upper", "lower"):
        z = norm.ppf(1.0 - alpha)
    else:
        raise ValueError("tail must be 'upper', 'lower', or 'two'")

    # thresholds per feature
    lows  = np.full(n_features, -np.inf, dtype=float)
    highs = np.full(n_features,  np.inf, dtype=float)

    if method.lower() == "zscore":
        center = np.nanmean(Xh, axis=0)
        scale  = np.nanstd(Xh, axis=0)
        scale  = np.maximum(scale, eps)

    elif method.lower() == "mad":
        center = np.nanmedian(Xh, axis=0)
        scale  = median_abs_deviation(Xh, axis=0, nan_policy="omit", scale="normal")
        scale  = np.maximum(scale, eps)

    else:
        raise ValueError("method must be 'zscore' or 'mad'")

    # Build bounds
    if tail == "two":
        lows  = center - z * scale
        highs = center + z * scale
    elif tail == "upper":
        highs = center + z * scale
        # lows stays -inf
    elif tail == "lower":
        lows  = center - z * scale
        # highs stays +inf

    # Apply removal ONLY to healthy points:
    Xh_all = X_arr[healthy_pos]

    violate_low  = (Xh_all < lows)   if tail in ("two", "lower") else np.zeros_like(Xh_all, dtype=bool)
    violate_high = (Xh_all > highs)  if tail in ("two", "upper") else np.zeros_like(Xh_all, dtype=bool)

    # Union rule across features: remove if ANY feature violates
    # violate_any = np.any(violate_low | violate_high, axis=1)
    k = 2
    violate_any = np.sum(violate_high, axis=1) >= k

    remove_pos = healthy_pos[violate_any]
    keep_mask = np.ones(n_samples, dtype=bool)
    keep_mask[remove_pos] = False

    removed_indices = original_idx[remove_pos]
    kept_indices = original_idx[keep_mask]

    # Return same type as input where possible
    if hasattr(X, "iloc"):
        X_cleaned = X.iloc[keep_mask].copy()
    else:
        X_cleaned = X_arr[keep_mask]

    if hasattr(y, "iloc"):
        y_cleaned = y.iloc[keep_mask].copy()
    else:
        y_cleaned = y_arr[keep_mask]

    thresholds = {
        "method": method,
        "confidence": confidence,
        "tail": tail,
        "z": float(z),
        "low": lows,
        "high": highs,
        "center": center,
        "scale": scale
    }

    if verbose:
        print(f"Total samples: {n_samples}")
        print(f"Healthy samples (fit): {len(healthy_pos)}")
        print(f"Tail mode: {tail} @ {confidence*100:.1f}%")
        print(f"Removed healthy outliers: {len(removed_indices)} "
              f"({100.0 * len(removed_indices)/max(1,len(healthy_pos)):.2f}% of healthy)")
        print(f"Samples after cleaning: {len(kept_indices)}")

    return X_cleaned, y_cleaned, removed_indices, kept_indices, thresholds


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _get_X_y_and_index(X, y):
    """Return X_arr, y_arr, idx_arr, feat_names safely for pandas or numpy."""
    if hasattr(X, "to_numpy"):  # pandas DataFrame
        X_arr = X.to_numpy()
        idx_arr = X.index.to_numpy()
        feat_names = list(X.columns)
    else:
        X_arr = np.asarray(X)
        idx_arr = np.arange(len(X_arr))
        feat_names = None

    if hasattr(y, "to_numpy"):  # pandas Series
        y_arr = y.to_numpy()
    else:
        y_arr = np.asarray(y)

    return X_arr, y_arr, idx_arr, feat_names


def _indices_to_mask(idx_arr, removed_indices):
    """
    Convert 'original indices' into a boolean mask aligned to idx_arr.
    If removed_indices look like positional indices (0..n-1), fallback to that.
    """
    removed_indices = np.asarray(removed_indices)

    # Primary: treat as original index values
    mask = np.isin(idx_arr, removed_indices)

    # Fallback: if nothing matches, but removed_indices look like positions
    if (not mask.any()) and removed_indices.size > 0:
        n = len(idx_arr)
        if np.all((removed_indices >= 0) & (removed_indices < n)):
            mask = np.zeros(n, dtype=bool)
            mask[removed_indices.astype(int)] = True

    return mask


def compare_outlier_removal_2d(
    X, y,
    removed_idx_knn,
    removed_idx_stat,
    f0=0, f1=1,
    healthy_label=0,
    remove_only_healthy=True,
    show_legend=True
):
    """
    Compare two outlier-removal methods in 2D using two features:
      - KNN-based method (e.g., RNN/KNN outlier removal)
      - Statistical thresholds (99% confidence) method

    Parameters
    ----------
    X, y : original data BEFORE removal
    removed_idx_knn : array-like
        ORIGINAL indices removed by KNN method
    removed_idx_stat : array-like
        ORIGINAL indices removed by statistical method
    f0, f1 : int
        Feature indices (x and y axis)
    healthy_label : int
        Label for healthy class (default 0)
    remove_only_healthy : bool
        If True, only highlight removals among healthy samples in the plot.
        (Fault samples are shown but never considered “removed” visually.)
    """

    X_arr, y_arr, idx_arr, feat_names = _get_X_y_and_index(X, y)

    x0 = X_arr[:, f0]
    x1 = X_arr[:, f1]

    label_x = feat_names[f0] if feat_names is not None else f"Feature {f0}"
    label_y = feat_names[f1] if feat_names is not None else f"Feature {f1}"

    # Removal masks aligned to original X
    rm_knn_all  = _indices_to_mask(idx_arr, removed_idx_knn)
    rm_stat_all = _indices_to_mask(idx_arr, removed_idx_stat)

    # Optionally consider only healthy removals (your use-case)
    healthy_mask = (y_arr == healthy_label)
    if remove_only_healthy:
        rm_knn  = rm_knn_all  & healthy_mask
        rm_stat = rm_stat_all & healthy_mask
    else:
        rm_knn  = rm_knn_all
        rm_stat = rm_stat_all

    keep_knn  = ~rm_knn
    keep_stat = ~rm_stat

    # Shared axis limits for fair visual comparison
    x_min, x_max = np.nanmin(x0), np.nanmax(x0)
    y_min, y_max = np.nanmin(x1), np.nanmax(x1)
    pad_x = 0.05 * (x_max - x_min + 1e-12)
    pad_y = 0.05 * (y_max - y_min + 1e-12)

    xlim = (x_min - pad_x, x_max + pad_x)
    ylim = (y_min - pad_y, y_max + pad_y)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

    # -----------------------------
    # Helper to draw one method
    # -----------------------------
    def draw(ax_before, ax_after, rm_mask, keep_mask, method_name):
        # BEFORE: show all kept points in gray, removed points in red, faults in blue outline (optional)
        ax_before.scatter(
            x0[keep_mask], x1[keep_mask],
            c="lightgray", alpha=0.6, s=40, label="Kept"
        )
        ax_before.scatter(
            x0[rm_mask], x1[rm_mask],
            c="red", edgecolors="k", alpha=0.9, s=70, label="Removed"
        )

        # Overlay faults as markers (not removed)
        fault_mask = ~healthy_mask
        ax_before.scatter(
            x0[fault_mask], x1[fault_mask],
            facecolors="none", edgecolors="tab:blue",
            alpha=0.9, s=70, linewidths=1.5, label="Fault (kept)"
        )

        n_removed = int(np.sum(rm_mask))
        n_kept = int(np.sum(keep_mask))
        ax_before.set_title(f"{method_name} — Before\nRemoved: {n_removed}, Kept: {n_kept}")
        ax_before.set_xlabel(label_x)
        ax_before.set_ylabel(label_y)
        ax_before.set_xlim(*xlim)
        ax_before.set_ylim(*ylim)

        # AFTER: color by class for the kept points only
        sc = ax_after.scatter(
            x0[keep_mask], x1[keep_mask],
            c=y_arr[keep_mask],
            cmap="coolwarm",
            alpha=0.75, s=45
        )
        ax_after.set_title(f"{method_name} — After (kept points)")
        ax_after.set_xlabel(label_x)
        ax_after.set_ylabel(label_y)
        ax_after.set_xlim(*xlim)
        ax_after.set_ylim(*ylim)

        return sc

    # Top row: KNN
    sc1 = draw(axes[0, 0], axes[0, 1], rm_knn, keep_knn, "KNN-based Outlier Removal")
    # Bottom row: Statistical
    sc2 = draw(axes[1, 0], axes[1, 1], rm_stat, keep_stat, "Statistical 99% (Healthy-fit)")

    if show_legend:
        # Use one legend for the whole figure (from first "before" axis)
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)


    title_note = "Removed points shown for healthy only" if remove_only_healthy else "Removed points shown for all classes"
    fig.suptitle(f"Outlier Removal Comparison (2D): {title_note}", y=0.98, fontsize=14)

    plt.tight_layout()
    plt.show()


In [ ]:
from collections import Counter
# Example usage
y_ds = y_train.values
X_ds = X_train
k_outlier = 10
print(f"Original dataset: {X_ds.shape[0]} samples")
print(f"Class distribution: {Counter(y_ds)}")
print()

# Apply complete preprocessing
X_border, y_border, out_idx, kept_idx, final_idx = complete_preprocessing(
    X_ds, y_ds, k_outlier=10, k_borderline=100
)

print(f"\nFinal class distribution: {Counter(y_border)}")

#### Use the KNN as Outlier Removal

In [ ]:
X_train_clean, y_train_clean, out_idx_knn, kept_idx_knn = reverse_nearest_neighbors_outlier_detection(
    X_train, y_train, k=5
)

X_cleaned, y_cleaned, out_idx_stats, kept_idx_stats, thresholds = statistical_outlier_removal(
    X_train, y_train,
    healthy_label=0,
    confidence=0.95,
    method="mad",
    tail="upper",     # <-- only remove high end
    verbose=True
    )
# X_train_clean and y_train_clean keep the original indices from df_filt
# out_idx tells you which original rows were flagged as outliers.
# Malfunction labels for the *clean* training points:
mal_train_clean = mal_train.loc[kept_idx_knn] 
# Then for visualization (before/after):

compare_outlier_removal_2d(
    X_train, y_train,
    removed_idx_knn=out_idx_knn,
    removed_idx_stat=out_idx_stats,
    f0=0, f1=1,
    healthy_label=0,
    remove_only_healthy=True
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import skew, kurtosis, shapiro, normaltest

for feature in selected_features:
    plt.figure(figsize=(6,4))
    sns.histplot(X_cleaned[feature], kde=True, bins=30, color="steelblue")
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.show()
    

# Create a results dictionary
results = {}

for feature in selected_features:
    data = X_cleaned[feature].dropna()  # drop NaNs if any
    
    # Metrics
    skewness = skew(data)
    kurt = kurtosis(data)
    
    results[feature] = {
        "Skewness": skewness,
        "Kurtosis": kurt,
    }

# Convert to DataFrame for readability
metrics_df = pd.DataFrame(results).T
print(metrics_df)

### Effect of Feature Scaling

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
## matplotlib inline
matplotlib.style.use('fivethirtyeight')
df_plot = df.copy()
x = X_train[selected_features]
robust_df = pd.DataFrame(preprocessing.RobustScaler().fit_transform(x), 
                         columns=selected_features)
standard_df = pd.DataFrame(preprocessing.StandardScaler().fit_transform(x), 
                           columns=selected_features)
minmax_df = pd.DataFrame(preprocessing.MinMaxScaler().fit_transform(x), 
                         columns=selected_features)

fig, axes = plt.subplots(ncols=4, figsize=(20, 5))
datasets = [x, robust_df, standard_df, minmax_df]
titles = ['Before Scaling', 'Robust Scaling', 'Standard Scaling', 'Min-Max Scaling']
colors = ['r', 'b', 'g', 'cyan', 'orange', 'purple']  # extend for more features

for ax, df_plot, title in zip(axes, datasets, titles):
    ax.set_title(title)
    for i, feature in enumerate(df_plot.columns):
        sns.kdeplot(df_plot[feature], ax=ax, color=colors[i % len(colors)], label=feature)
    ax.legend()
plt.show()

In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

# 0) Make sure X_train is a DataFrame (recommended)
# If X_train is numpy, keep separate index array instead.
# KNN kept / removed masks using original indices
X_train_knn_clean = X_train.loc[kept_idx_knn]
y_train_knn_clean = y_train.loc[kept_idx_knn]

X_train_knn_removed = X_train.loc[out_idx_knn]
y_train_knn_removed = y_train.loc[out_idx_knn]

# Stats kept / removed
X_train_stat_clean = X_train.loc[kept_idx_stats]
y_train_stat_clean = y_train.loc[kept_idx_stats]

X_train_stat_removed = X_train.loc[out_idx_stats]
y_train_stat_removed = y_train.loc[out_idx_stats]

X_train_use = X_train_knn_clean
y_train_use = y_train_knn_clean

[X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer] = scale_features(X_train_use, X_test, X_train_healthy)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

# Choose reference for PCA fitting
X_pca_fit = X_train_use  # healthy-only reference

# 1) Fit imputer on reference, transform all sets
imputer = SimpleImputer(strategy="median")
X_fit_imp = imputer.fit_transform(X_pca_fit)

X_all_imp        = imputer.transform(X_train)
X_knn_clean_imp  = imputer.transform(X_train_knn_clean)
X_knn_rem_imp    = imputer.transform(X_train_knn_removed)
X_stat_clean_imp = imputer.transform(X_train_stat_clean)
X_stat_rem_imp   = imputer.transform(X_train_stat_removed)

# 2) Fit scaler on reference, transform all sets
scaler = RobustScaler()
X_fit_scaled = scaler.fit_transform(X_fit_imp)

X_all_scaled        = scaler.transform(X_all_imp)
X_knn_clean_scaled  = scaler.transform(X_knn_clean_imp)
X_knn_rem_scaled    = scaler.transform(X_knn_rem_imp)
X_stat_clean_scaled = scaler.transform(X_stat_clean_imp)
X_stat_rem_scaled   = scaler.transform(X_stat_rem_imp)

# 3) Fit PCA on reference, transform all sets
pca = PCA(n_components=2, random_state=42)
pca.fit(X_fit_scaled)

Z_all        = pca.transform(X_all_scaled)
Z_knn_clean  = pca.transform(X_knn_clean_scaled)
Z_knn_rem    = pca.transform(X_knn_rem_scaled)
Z_stat_clean = pca.transform(X_stat_clean_scaled)
Z_stat_rem   = pca.transform(X_stat_rem_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Cumulative:", pca.explained_variance_ratio_.cumsum())


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import numpy as np

def plot_pca_removal_comparison(
    Z_all, y_all,
    Z_clean, y_clean,
    Z_removed, y_removed,
    title
):
    fig, ax = plt.subplots(figsize=(8, 6))

    # Background: all training points (faded)
    ax.scatter(Z_all[:,0], Z_all[:,1], c="lightgray", alpha=0.35, s=25, label="All train")
    cmap = ListedColormap(["tab:blue", "tab:orange", "tab:red"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

    ax.scatter(
        Z_clean[:,0], Z_clean[:,1],
        c=y_clean,
        cmap=cmap,
        norm=norm,
        alpha=0.8,
        s=35
    )
    # Kept points: colored by class label
    sc = ax.scatter(Z_clean[:,0], Z_clean[:,1], c=y_clean, cmap="coolwarm",
                    alpha=0.8, s=35, label="Kept (colored by y)")

    # Removed points: red edge markers
    if len(Z_removed) > 0:
        ax.scatter(Z_removed[:,0], Z_removed[:,1], facecolors="none", edgecolors="red",
                   s=90, linewidths=1.8, label="Removed")
    print("Unique y_clean:", np.unique(y_clean))
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    # cbar = plt.colorbar(sc, ax=ax, ticks=[0, 1, 2])
    # cbar.set_label("Class label")

    ax.legend()
    plt.tight_layout()
    plt.show()

# y arrays
y_all_arr        = np.asarray(y_train)
y_knn_clean_arr  = np.asarray(y_train_knn_clean)
y_knn_rem_arr    = np.asarray(y_train_knn_removed)
y_stat_clean_arr = np.asarray(y_train_stat_clean)
y_stat_rem_arr   = np.asarray(y_train_stat_removed)

plot_pca_removal_comparison(
    Z_all, y_all_arr,
    Z_knn_clean, y_knn_clean_arr,
    Z_knn_rem, y_knn_rem_arr,
    title="PCA (fit on healthy) — KNN outlier removal"
)

plot_pca_removal_comparison(
    Z_all, y_all_arr,
    Z_stat_clean, y_stat_clean_arr,
    Z_stat_rem, y_stat_rem_arr,
    title="PCA (fit on healthy) — Statistical upper 99% removal"
)
print("Removed by KNN:", len(out_idx_knn))
print("Removed by Stat:", len(out_idx_stats))
print("Overlap removed:", len(set(out_idx_knn).intersection(set(out_idx_stats))))


### SMOTE

#### Differences between SMOTE Function

In [ ]:
from collections import Counter
import pandas as pd
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN, KMeansSMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# ---------------------------------------------------------
# BASE DATA
# ---------------------------------------------------------
X = X_train_scaled
y = y_train_use
sampling_ratio = 0.5

# Wrap in DataFrame for consistent columns
if isinstance(X, pd.DataFrame):
    feature_names = X.columns.tolist()
    X_df = X.copy()
else:
    feature_names = [f"feat_{i}" for i in range(X.shape[1])]
    X_df = pd.DataFrame(X, columns=feature_names)

y_ser = pd.Series(y, name="label")

counts = Counter(y_ser)
n_maj = max(counts.values()) # majority count

target = {
    1: int(0.5 * n_maj),
    2: int(0.5 * n_maj),
    # do NOT include 0 (majority) in oversampling dict
}
print("Before oversampling:", counts)
# ---------------------------------------------------------
# DICTIONARY OF ALL SAMPLERS
# ---------------------------------------------------------
oversamplers = {
    "SMOTE": SMOTE(sampling_strategy=target, random_state=42),
    "BorderlineSMOTE": BorderlineSMOTE(sampling_strategy=target, random_state=42),
    "SMOTE-Tomek": SMOTETomek(sampling_strategy=target, random_state=42),
    "SMOTE-ENN": SMOTEENN(sampling_strategy=target, random_state=42),
    "ADASYN": ADASYN(sampling_strategy=target, random_state=42),
    "KMeans-SMOTE": KMeansSMOTE(sampling_strategy=target,cluster_balance_threshold=0.05, random_state=42)
}

# ---------------------------------------------------------
# CREATE THE MAIN DICTIONARY OF ALL RESULTS
# ---------------------------------------------------------
resampled_data = {}   # MASTER DICT
summary_rows = []



for name, sampler in oversamplers.items():
    pipeline = Pipeline([
        ("sampler", sampler)
    ])
    
    X_res, y_res = pipeline.fit_resample(X_df, y_ser)

    # store in a structured dict
    resampled_data[name] = {
        "X": pd.DataFrame(X_res, columns=feature_names),
        "y": pd.Series(y_res, name="label"),
        "sampler": sampler
    }

    # Compute useful statistics
    cnt = Counter(y_res)
    summary_rows.append({
        "Sampler": name,
        "Total_samples": len(y_res),
        "Class_0_count": cnt.get(0, 0),
        "Class_1_count": cnt.get(1, 0),
        "Class_2_count": cnt.get(2, 0),
        "Minority_class": min(cnt, key=cnt.get),
        "Majority_class": max(cnt, key=cnt.get),
        "Imbalance_ratio": max(cnt.values()) / min(cnt.values())
    })

# Add original dataset for comparison
cnt0 = Counter(y_ser)
summary_rows.insert(0, {
    "Sampler": "Original",
    "Total_samples": len(y_ser),
    "Class_0_count": cnt0.get(0, 0),
    "Class_1_count": cnt0.get(1, 0),
    "Class_2_count": cnt0.get(2, 0),
    "Minority_class": min(cnt0, key=cnt0.get),
    "Majority_class": max(cnt0, key=cnt0.get),
    "Imbalance_ratio": max(cnt0.values()) / min(cnt0.values())
})

resample_summary_df = pd.DataFrame(summary_rows)
resample_summary_df


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import BoundaryNorm

# ------------------------------------------------------------
# Ensure X is a DataFrame
# ------------------------------------------------------------
if isinstance(X_train_scaled, pd.DataFrame):
    X_orig = X_train_scaled.copy()
    feature_names = X_orig.columns.tolist()
else:
    X_orig = pd.DataFrame(X_train_scaled)
    feature_names = X_orig.columns.tolist()

# IMPORTANT: make sure y is aligned in length to X
y_orig = pd.Series(y_train_use, name="label").reset_index(drop=True)
X_orig = X_orig.reset_index(drop=True)

if len(y_orig) != len(X_orig):
    raise ValueError(f"Length mismatch: X_orig={len(X_orig)} vs y_orig={len(y_orig)}. "
                     "Make sure y_train corresponds to X_train_scaled rows.")

# ------------------------------------------------------------
# FUNCTION: Scatterplot before vs after SMOTE/ADASYN/etc.
# ------------------------------------------------------------
def scatter_compare_by_index(
    X_orig, y_orig,
    X_res, y_res,
    idx_x, idx_y,
    sampler_name,
    selected_features=None
):
    # Ensure resampled are DataFrame/Series
    if not isinstance(X_res, pd.DataFrame):
        X_res = pd.DataFrame(X_res, columns=X_orig.columns)
    if not isinstance(y_res, pd.Series):
        y_res = pd.Series(y_res, name="label")

    X_res = X_res.reset_index(drop=True)
    y_res = y_res.reset_index(drop=True)

    # Feature labels
    if selected_features is not None:
        feat_x = selected_features[idx_x]
        feat_y = selected_features[idx_y]
    else:
        feat_x = feature_names[idx_x]
        feat_y = feature_names[idx_y]

    # ---- Discrete colormap for multiclass (0/1/2) ----
    classes = np.sort(np.unique(pd.concat([y_orig.dropna(), y_res.dropna()]).astype(int)))

    # Pick 3 clearly distinct colors (edit if you want)
    color_map = {
        0: "tab:blue",
        1: "tab:orange",
        2: "tab:red",
    }

    colormap = ListedColormap([color_map[c] for c in classes])

    # boundaries like [-0.5, 0.5, 1.5, 2.5]
    boundaries = np.r_[classes - 0.5, classes[-1] + 0.5]
    norm = BoundaryNorm(boundaries, ncolors=len(classes))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # BEFORE
    sc0 = axes[0].scatter(
        X_orig.iloc[:, idx_x],
        X_orig.iloc[:, idx_y],
        c=y_orig.astype(int),
        cmap=colormap,
        norm=norm,
        alpha=0.6, s=20, edgecolors="none"
    )
    axes[0].set_title("Original Scaled", fontsize=16, fontweight="bold")
    axes[0].set_xlabel(feat_x, fontsize=14)
    axes[0].set_ylabel(feat_y, fontsize=14)
    axes[0].tick_params(labelsize=12)

    # AFTER
    sc1 = axes[1].scatter(
        X_res.iloc[:, idx_x],
        X_res.iloc[:, idx_y],
        c=y_res.astype(int),
        cmap=colormap,
        norm=norm,
        alpha=0.6, s=20, edgecolors="none"
    )
    axes[1].set_title(f"{sampler_name}", fontsize=16, fontweight="bold")
    axes[1].set_xlabel(feat_x, fontsize=14)
    axes[1].set_ylabel(feat_y, fontsize=14)
    axes[1].tick_params(labelsize=12)

    # Optional: match axis limits automatically (no clipping)
    x_min = min(X_orig.iloc[:, idx_x].min(), X_res.iloc[:, idx_x].min())
    x_max = max(X_orig.iloc[:, idx_x].max(), X_res.iloc[:, idx_x].max())
    y_min = min(X_orig.iloc[:, idx_y].min(), X_res.iloc[:, idx_y].min())
    y_max = max(X_orig.iloc[:, idx_y].max(), X_res.iloc[:, idx_y].max())
    pad_x = 0.05 * (x_max - x_min + 1e-9)
    pad_y = 0.05 * (y_max - y_min + 1e-9)
    axes[0].set_xlim([x_min - pad_x, x_max + pad_x])
    axes[1].set_xlim([x_min - pad_x, x_max + pad_x])
    axes[0].set_ylim([y_min - pad_y, y_max + pad_y])
    axes[1].set_ylim([y_min - pad_y, y_max + pad_y])

    # Legend-like colorbar with class ticks
    # cbar = fig.colorbar(sc1, ax=axes.ravel().tolist(), ticks=classes)
    # cbar.set_label("Class", fontsize=12)

    fig.tight_layout()
    plt.show()


# ------------------------------------------------------------
# CHOOSE FEATURE INDEX PAIRS TO PLOT
# ------------------------------------------------------------
feature_pairs_idx = [
    (0, 1),
]

# ------------------------------------------------------------
# LOOP THROUGH ALL RESAMPLERS AND PLOT SCATTERS
# ------------------------------------------------------------
for sampler_name, data_dict in resampled_data.items():
    if sampler_name.lower() == "original":
        continue

    X_res_df = data_dict["X"]
    y_res_ser = data_dict["y"]

    print(f"\n=== Scatterplots for {sampler_name} ===")

    for idx_x, idx_y in feature_pairs_idx:
        scatter_compare_by_index(
            X_orig, y_orig,
            X_res_df, y_res_ser,
            idx_x, idx_y,
            sampler_name,
            selected_features=selected_features
        )


## Model Definition and Helper Function

### Defining function for Model used, Train Model, and Cross Validations

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE


def get_all_models(X_train):
    """
    Return ONLY supervised models intended for training on SMOTE-resampled data.
    """
    n_features = X_train.shape[1]

    models = {
        'supervised_smote': {
            'Random Forest (SMOTE)': RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                max_depth=n_features,
                n_jobs=-1
            ),

            'XGBoost (SMOTE)': XGBClassifier(
                n_estimators=200,
                random_state=42,
                max_depth=n_features,
                objective='multi:softprob',
                num_class=3,
                eval_metric='mlogloss'
            ),

            'Logistic Regression (SMOTE)': LogisticRegression(
                C=0.1,
                random_state=42,
                max_iter=1000,
                solver='lbfgs',          # or 'saga'
                multi_class='multinomial'
            ),


            'Decision Tree (SMOTE)': DecisionTreeClassifier(
                random_state=42,
                max_depth=n_features,
                max_features=n_features,
                min_samples_leaf=2,
                min_samples_split=2
            ),

            'KNN (SMOTE)': KNeighborsClassifier(
                n_neighbors=15,
                weights='distance',
                metric='euclidean',
                p=1
            ),

            'SVM (SMOTE)': SVC(
                kernel='linear',        # common choice, can be 'linear' too
                C=1.0,               # regularization strength
                gamma='scale',       # auto scaling of kernel coefficient
                probability=True,    # enables predict_proba for ROC/AUC
                random_state=42
            ),
            
            'SVM_rbf (SMOTE)': SVC(
                kernel='rbf',        # common choice, can be 'linear' too
                C=1.0,               # regularization strength
                gamma='scale',       # auto scaling of kernel coefficient
                probability=True,    # enables predict_proba for ROC/AUC
                random_state=42
            )
        }
    }

    return models

In [ ]:
def train_all_models(X_train, y_train, model_group='supervised_smote'):
    """
    Train a group of models on the *already prepared* training data
    (e.g., after SMOTE / ADASYN / KMeansSMOTE, or even original).

    Parameters
    ----------
    X_train : array-like or DataFrame
        Training features (can already be oversampled and scaled).
    y_train : array-like or Series
        Training labels (aligned with X_train).
    model_group : str
        Which group from get_all_models() to use, e.g.:
        - 'supervised'        : if you trained on original data
        - 'supervised_smote'  : if you conceptually group these as SMOTE-based models
        - any other key you defined in get_all_models()

    Returns
    -------
    trained_models : dict
        { model_name: {'model': estimator, 'type': model_group, 'trained': True} }
    """

    models = get_all_models(X_train)          # your existing function
    if model_group not in models:
        raise ValueError(
            f"Model group '{model_group}' not found in get_all_models(). "
            f"Available groups: {list(models.keys())}"
        )

    model_dict = models[model_group]
    trained_models = {}

    print("\n" + "="*60)
    print(f"TRAINING MODELS ({model_group}) ON PROVIDED DATA")
    print("="*60)
    print(f"  - X_train shape: {X_train.shape}")
    y_arr = np.asarray(y_train)
    print("  - y_train distribution: "
        f"0={(y_arr==0).sum()}, 1={(y_arr==1).sum()}, 2={(y_arr==2).sum()}")


    for name, model in model_dict.items():
        print(f"  - Training {name}...")
        model.fit(X_train, y_train)
        trained_models[name] = {
            'model': model,
            'type': model_group,
            'trained': True
        }

    print("\nAll models trained successfully!")
    return trained_models


In [ ]:
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN, KMeansSMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
import numpy as np

def get_oversampler(name="SMOTE", y=None, n_splits=5, sampling_ratio=None, random_state=42):
    """
    Oversampler that auto-reduces k_neighbors (or n_neighbors) so it does not crash inside CV folds.

    Requirements:
    - pass y (labels) and n_splits (your CV folds count)
    """

    if y is None:
        raise ValueError("Pass y to get_oversampler so k_neighbors can be chosen safely.")

    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    min_count = counts.min()

    # Minimum samples of the smallest class available in a TRAINING fold:
    # train fraction = (n_splits - 1) / n_splits
    min_train_fold = int(np.floor(min_count * (n_splits - 1) / n_splits))

    # k_neighbors must be <= (min_train_fold - 1)
    # keep it at most 5 (classic default), but shrink if needed
    safe_k = max(1, min(5, min_train_fold - 1))

    if sampling_ratio is None:
        sampling_strategy = "not majority"
    else:
        sampling_strategy = sampling_ratio

    name_l = name.lower()

    if name_l == "smote":
        return SMOTE(sampling_strategy=sampling_strategy, k_neighbors=safe_k, random_state=random_state)

    elif name_l == "borderlinesmote":
        return BorderlineSMOTE(sampling_strategy=sampling_strategy, k_neighbors=safe_k, random_state=random_state)

    elif name_l == "adasyn":
        # ADASYN uses n_neighbors (validated KNN estimator) :contentReference[oaicite:1]{index=1}
        return ADASYN(sampling_strategy=sampling_strategy, n_neighbors=safe_k, random_state=random_state)

    elif name_l in ("kmeans-smote", "kmeanssmote"):
        return KMeansSMOTE(sampling_strategy=sampling_strategy, k_neighbors=safe_k, random_state=random_state)

    elif name_l in ("smote-tomek", "smotetomek"):
        # SMOTETomek takes an internal SMOTE object
        return SMOTETomek(
            sampling_strategy=sampling_strategy,
            smote=SMOTE(sampling_strategy=sampling_strategy, k_neighbors=safe_k, random_state=random_state),
            random_state=random_state
        )

    elif name_l in ("smote-enn", "smoteenn"):
        return SMOTEENN(
            sampling_strategy=sampling_strategy,
            smote=SMOTE(sampling_strategy=sampling_strategy, k_neighbors=safe_k, random_state=random_state),
            random_state=random_state
        )

    else:
        raise ValueError(f"Unknown oversampler '{name}'.")


### CROSS VALIDATION METHOD


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, balanced_accuracy_score
)
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
import pandas as pd
import numpy as np


def crossval_metrics(
    X,
    y,
    model_group="supervised_smote",     # <- key: pulls models from get_all_models(X)
    use_smote_in_cv=True,
    oversampler_name="SMOTE",
    sampling_ratio=None,               # dict / float / 'auto' / etc.
    average="macro",                   # macro / weighted / micro
    multiclass_roc_auc="ovr",      # 'ovr' or 'ovo' for multiclass ROC-AUC
    n_splits = 5, 
    random_state=42
):
    """
    Cross-validated metrics for ALL models returned by get_all_models(X)[model_group].

    Notes
    -----
    - CV always refits models per fold; this function does not need pre-trained models.
    - ROC-AUC is computed only if predict_proba is available.
      For SVC, set probability=True if you need ROC-AUC.
    """

    # --- safety: ensure arrays ---
    X = np.asarray(X)
    y = np.asarray(y)

    # Find the minimum class count
    classes, counts = np.unique(y, return_counts=True)
    min_count = counts.min()
    
    # Calculate safe k_neighbors and n_splits
    # k_neighbors must be < min_count, and we need buffer for CV splits
    # In CV, a fold might have only (n_splits-1)/n_splits of the data
    min_samples_in_fold = int(min_count * (n_splits - 1) / n_splits)
    
    # Set k_neighbors conservatively: at least 1, at most min_samples_in_fold - 1
    k = max(1, min(5, min_samples_in_fold - 1))
    
    # Also ensure n_splits doesn't exceed min_count
    n_splits = min(n_splits, min_count)
    
    print(f"Class distribution: {dict(zip(classes, counts))}")
    print(f"Using k_neighbors={k}, n_splits={n_splits}")
    
    # Create CV strategy
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    is_multiclass = (len(classes) > 2)

    # --- get models ---
    all_models = get_all_models(X)
    if model_group not in all_models:
        raise ValueError(
            f"Model group '{model_group}' not found in get_all_models(). "
            f"Available groups: {list(all_models.keys())}"
        )

    model_dict = all_models[model_group]

    rows = []

    for name, base_model in model_dict.items():

        # 1) Build estimator for CV (oversampler INSIDE fold, to avoid leakage)
        if use_smote_in_cv:
            oversampler = get_oversampler(
                name=oversampler_name,
                y=y,
                n_splits=n_splits,
                sampling_ratio=sampling_ratio,
                random_state=random_state
            )
            estimator = ImbPipeline([
                ("oversampler", oversampler),
                ("clf", clone(base_model))
            ])
        else:
            estimator = clone(base_model)

        # 2) Out-of-fold predictions
        y_pred = cross_val_predict(estimator, X, y, cv=cv, method="predict")

        # 3) Metrics (label-based)
        cm = confusion_matrix(y, y_pred, labels=classes)

        precision = precision_score(y, y_pred, average=average, zero_division=0)
        recall    = recall_score(y, y_pred, average=average, zero_division=0)
        f1        = f1_score(y, y_pred, average=average, zero_division=0)
        bal_acc   = balanced_accuracy_score(y, y_pred)

        # 4) ROC-AUC (probability-based, optional)
        rocauc = np.nan
        try:
            y_proba = cross_val_predict(estimator, X, y, cv=cv, method="predict_proba")

            if not is_multiclass:
                # binary: pick probability of the "positive" class.
                # Here we assume class '1' is the positive class if it exists; otherwise use the last class.
                pos_label = 1 if 1 in classes else classes[-1]
                pos_idx = np.where(classes == pos_label)[0][0]
                rocauc = roc_auc_score(y, y_proba[:, pos_idx])

            else:
                rocauc = roc_auc_score(
                    y, y_proba,
                    multi_class=multiclass_roc_auc,
                    average=average
                )

        except Exception:
            # e.g., SVC(probability=False) has no predict_proba
            y_proba = None

        rows.append({
            "Model": name,
            "Type": model_group,
            "UseOversamplerInCV": use_smote_in_cv,
            "Oversampler": oversampler_name if use_smote_in_cv else "None",
            "SamplingRatio": sampling_ratio,
            "Average": average,
            "n_splits": n_splits,
            "BalancedAcc": bal_acc,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "ConfusionMatrix": cm
        })

    df = pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
    return df


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, balanced_accuracy_score
)
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
import pandas as pd
import numpy as np
from collections import Counter

def far_from_cm_multiclass(cm, classes, healthy_label=0):
    """
    FAR for multiclass: among true-healthy samples, how many were predicted as any fault class?
    FAR = FP / (FP + TN), where:
      TN = CM[healthy, healthy]
      FP = sum_{pred != healthy} CM[healthy, pred]
    """
    if healthy_label not in classes:
        return np.nan

    h = np.where(classes == healthy_label)[0][0]
    TN = cm[h, h]
    FP = cm[h, :].sum() - TN
    denom = TN + FP
    return FP / denom if denom > 0 else np.nan

def multiclass_sampling_dict(y, majority_class=0, ratio=0.5):
    """
    Build a sampling_strategy dict for multiclass oversampling.
    Each minority class is oversampled up to ratio * n_majority.
    """
    counts = Counter(y)
    n_maj = counts[majority_class]
    target = int(np.floor(ratio * n_maj))

    strategy = {}
    for cls, n in counts.items():
        if cls == majority_class:
            continue
        # Only oversample if it increases the class count
        if n < target:
            strategy[cls] = target
    return strategy


def crossval_metrics_repeated(
    X,
    y,
    model_group="supervised_smote",
    use_smote_in_cv=True,
    oversampler_name="SMOTE",
    sampling_ratio=None,
    average="macro",
    multiclass_roc_auc="ovr",
    n_splits=5,
    random_state=42,
    n_repeats=10,                 # <-- NEW
    seed_step=1000                # <-- NEW: spacing between seeds
):
    """
    Repeated Stratified K-Fold CV with out-of-fold predictions per repeat.
    Reports mean/std across repeats (captures split sensitivity).
    """

    X = np.asarray(X)
    y = np.asarray(y)

    classes, counts = np.unique(y, return_counts=True)
    min_count = counts.min()

    n_splits = min(n_splits, min_count)
    min_samples_in_fold = int(min_count * (n_splits - 1) / n_splits)
    k = max(1, min(5, min_samples_in_fold - 1))

    print(f"Class distribution: {dict(zip(classes, counts))}")
    print(f"Using k_neighbors={k}, n_splits={n_splits}, n_repeats={n_repeats}")

    is_multiclass = (len(classes) > 2)

    all_models = get_all_models(X)
    if model_group not in all_models:
        raise ValueError(
            f"Model group '{model_group}' not found. "
            f"Available groups: {list(all_models.keys())}"
        )

    model_dict = all_models[model_group]

    rows = []

    for name, base_model in model_dict.items():

        # Collect metrics per repeat
        rep_metrics = []

        for r in range(n_repeats):
            seed_r = random_state + r * seed_step

            cv = StratifiedKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=seed_r
            )

            # Build estimator (oversampler inside fold)
            if use_smote_in_cv:
                strategy = multiclass_sampling_dict(y, majority_class=0, ratio=sampling_ratio)

                oversampler = get_oversampler(
                    name=oversampler_name,
                    y=y,
                    n_splits=n_splits,
                    sampling_ratio=strategy,     # <-- pass dict here
                    random_state=seed_r,
                )

                estimator = ImbPipeline([
                    ("oversampler", oversampler),
                    ("clf", clone(base_model))
                ])
            else:
                estimator = clone(base_model)

            # OOF predictions for this repeat
            y_pred = cross_val_predict(estimator, X, y, cv=cv, method="predict")

            cm = confusion_matrix(y, y_pred, labels=classes)
            far = far_from_cm_multiclass(cm, classes, healthy_label=0)

            precision = precision_score(y, y_pred, average=average, zero_division=0)
            recall    = recall_score(y, y_pred, average=average, zero_division=0)
            f1        = f1_score(y, y_pred, average=average, zero_division=0)
            bal_acc   = balanced_accuracy_score(y, y_pred)

            rocauc = np.nan
            try:
                y_proba = cross_val_predict(estimator, X, y, cv=cv, method="predict_proba")
                if not is_multiclass:
                    pos_label = 1 if 1 in classes else classes[-1]
                    pos_idx = np.where(classes == pos_label)[0][0]
                    rocauc = roc_auc_score(y, y_proba[:, pos_idx])
                else:
                    rocauc = roc_auc_score(
                        y, y_proba,
                        multi_class=multiclass_roc_auc,
                        average=average
                    )
            except Exception:
                pass

            rep_metrics.append({
                "BalancedAcc": bal_acc,
                "Precision": precision,
                "Recall": recall,
                "F1-Score": f1,
                "ROC-AUC": rocauc,
                "FAR": far,
                "ConfusionMatrix": cm
            })

        # Aggregate over repeats
        df_rep = pd.DataFrame(rep_metrics)

        rows.append({
            "Model": name,
            "Type": model_group,
            "UseOversamplerInCV": use_smote_in_cv,
            "Oversampler": oversampler_name if use_smote_in_cv else "None",
            "SamplingRatio": sampling_ratio,
            "Average": average,
            "n_splits": n_splits,
            "n_repeats": n_repeats,

            "BalancedAcc_mean": df_rep["BalancedAcc"].mean(),
            "BalancedAcc_std":  df_rep["BalancedAcc"].std(ddof=1),

            "Precision_mean": df_rep["Precision"].mean(),
            "Precision_std":  df_rep["Precision"].std(ddof=1),

            "Recall_mean": df_rep["Recall"].mean(),
            "Recall_std":  df_rep["Recall"].std(ddof=1),

            "F1_mean": df_rep["F1-Score"].mean(),
            "F1_std":  df_rep["F1-Score"].std(ddof=1),

            "ROC-AUC_mean": df_rep["ROC-AUC"].mean(),
            "ROC-AUC_std":  df_rep["ROC-AUC"].std(ddof=1),

            "FAR_mean": df_rep["FAR"].mean(),
            "FAR_std":  df_rep["FAR"].std(ddof=1),
  
            # Optional: keep per-repeat metrics for later plots
            "PerRepeat": rep_metrics
        })

    out = pd.DataFrame(rows).sort_values(
        by=["FAR_mean", "FAR_std", "Recall_mean"],
        ascending=[True, True, False]
    ).reset_index(drop=True)
    return out


### Model Hyperparameter Tuning

#### RANDOM FOREST 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_random_forest_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for Random Forest on already-resampled data.
    No class_weight, no SMOTE inside CV.

    Parameters
    ----------
    X_res, y_res : resampled training data (e.g. from SMOTE, ADASYN)
    sampler_name : optional string, just for printing ("SMOTE", "ADASYN", ...)

    Returns
    -------
    best_estimator_, best_params_, best_score_
    """

    rf_base = RandomForestClassifier(
        random_state=42,
        max_depth=X_res.shape[1],
        n_jobs=-1
    )

    param_grid = {
        'n_estimators':      [100, 200],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 3, 5],
        'max_features':      ['sqrt', 'log2'],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # cv = LeaveOneOut()

    print(f"\n===== Tuning Random Forest on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=rf_base,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best RF params:", grid.best_params_)
    print("Best RF CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### XGBOOST

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_xgboost_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for XGBoost on already-resampled data.
    Do NOT use scale_pos_weight here (class is already rebalanced).
    """

    # Base model (scale_pos_weight will be tuned)
    xgb_base = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42
    )


    # compute approximate neg/pos ratio:
    n_pos = (y_res == 1).sum()
    n_neg = (y_res == 0).sum()
    ratio = n_neg / max(n_pos, 1)

    param_grid = {
        "n_estimators": [100, 200],
        "learning_rate": [0.01, 0.1],
        # 'max_depth':         [2, 3, 4],
        "min_child_weight": [3, 4, 5],
        "subsample": [0.6, 0.8],
        "colsample_bytree": [0.6, 0.8],
        "gamma": [0.0, 0.1, 0.5],
        # tune around the empirical imbalance ratio
        "scale_pos_weight": [ratio],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # cv = LeaveOneOut()

    print(f"\n===== Tuning XGBoost on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=xgb_base,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best XGB params:", grid.best_params_)
    print("Best XGB CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### LOGISTIC REGRESSION

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_logistic_regression_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for Logistic Regression on already-resampled data.
    No class_weight, since oversampling already handled imbalance.
    """

    lr_base = LogisticRegression(
        solver="saga",
        multi_class="multinomial",
        max_iter=2000,
        random_state=42
    )
    param_grid = {"C": [0.001, 0.01, 0.1, 1, 10]}


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # cv = LeaveOneOut()

    print(f"\n===== Tuning Logistic Regression on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=lr_base,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best LR params:", grid.best_params_)
    print("Best LR CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### KNN TUNING

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_knn_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for KNN on already-resampled data.
    """

    knn_base = KNeighborsClassifier()

    param_grid = [
    {"metric": ["euclidean"], "n_neighbors": [5, 8, 10, 12], "weights":["distance","uniform"]},
    {"metric": ["manhattan"], "n_neighbors": [5, 8, 10, 12], "weights":["distance","uniform"]},
    {"metric": ["minkowski"], "p":[1,2], "n_neighbors":[5, 8, 10, 12], "weights":["distance","uniform"]}
    ]


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # cv = LeaveOneOut()
    
    print(f"\n===== Tuning KNN on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=knn_base,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best KNN params:", grid.best_params_)
    print("Best KNN CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### DECISION TREE TUNING

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_decision_tree_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for Decision Tree on already-resampled data.
    No class_weight here.
    """

    dt_base = DecisionTreeClassifier(
        class_weight="balanced", 
        max_depth=X_res.shape[1], 
        random_state=42
    )

    param_grid = {
        "criterion": ["gini", "entropy", "log_loss"],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [2, 3, 5],
        "max_features": ["sqrt", "log2", None],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print(f"\n===== Tuning Decision Tree on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=dt_base,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best DT params:", grid.best_params_)
    print("Best DT CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### SVM Linear SMOTE

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_svm_linear_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for LINEAR SVM on already-resampled data.
    No class_weight, no resampling inside CV.

    Parameters
    ----------
    X_res, y_res : resampled training data (e.g. from SMOTE, ADASYN)
    sampler_name : optional string for printing ("SMOTE", "ADASYN", ...)

    Returns
    -------
    best_estimator_, best_params_, best_score_, cv_results_
    """

    svm_linear = SVC(
        kernel="linear",
        probability=False,
        random_state=42
    )

    param_grid = {
        "C": [0.01, 0.1, 1, 10]
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print(f"\n===== Tuning LINEAR SVM on {sampler_name or 'resampled'} data =====")

    grid = GridSearchCV(
        estimator=svm_linear,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_res, y_res)

    print("Best Linear SVM params:", grid.best_params_)
    print("Best Linear SVM CV F1 :", grid.best_score_)

    return (
        grid.best_estimator_,
        grid.best_params_,
        grid.best_score_,
        grid.cv_results_
    )


#### SVM rbf SMOTE

In [ ]:
def tune_svm_rbf_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for RBF SVM on already-resampled data.
    No class_weight, no resampling inside CV.

    Parameters
    ----------
    X_res, y_res : resampled training data (e.g. from SMOTE, ADASYN)
    sampler_name : optional string for printing ("SMOTE", "ADASYN", ...)

    Returns
    -------
    best_estimator_, best_params_, best_score_, cv_results_
    """

    svm_rbf = SVC(
        kernel="rbf",
        probability=False,
        random_state=42
    )

    param_grid = {
        "C":     [0.1, 1, 10],
        "gamma": ["scale", 0.01, 0.1, 1]
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print(f"\n===== Tuning RBF SVM on {sampler_name or 'resampled'} data =====")

    grid = GridSearchCV(
        estimator=svm_rbf,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_res, y_res)

    print("Best RBF SVM params:", grid.best_params_)
    print("Best RBF SVM CV F1 :", grid.best_score_)

    return (
        grid.best_estimator_,
        grid.best_params_,
        grid.best_score_,
        grid.cv_results_
    )


### One Pipeline for Model Tuning

In [ ]:
from imblearn.base import BaseEstimator, FunctionSampler

def _rnn_clean_healthy_only(X, y, k=5):
    """
    Remove outliers from healthy class (y==0) only.
    Other classes are kept untouched.
    """
    X = np.asarray(X)
    y = np.asarray(y)

    healthy_mask = (y == 0)
    fault_mask = ~healthy_mask

    X_h = X[healthy_mask]
    y_h = y[healthy_mask]

    if len(X_h) == 0 or len(X_h) <= k:
        return X, y

    from sklearn.neighbors import NearestNeighbors

    n_h = len(X_h)
    reverse_neighbor_count = np.zeros(n_h, dtype=int)

    nbrs = NearestNeighbors(n_neighbors=min(k + 1, n_h))
    nbrs.fit(X_h)
    _, indices = nbrs.kneighbors(X_h)

    for neighbors in indices:
        for nb in neighbors[1:]:
            reverse_neighbor_count[nb] += 1

    keep_h = reverse_neighbor_count > 0

    X_h_clean = X_h[keep_h]
    y_h_clean = y_h[keep_h]

    X_res = np.vstack([X_h_clean, X[fault_mask]]) if X_h_clean.size else X[fault_mask]
    y_res = np.concatenate([y_h_clean, y[fault_mask]]) if y_h_clean.size else y[fault_mask]

    return X_res, y_res


def make_rnn_sampler(k=5):
    return FunctionSampler(func=_rnn_clean_healthy_only, kw_args={"k": k})

from sklearn.model_selection import GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold
def tune_one_model(
    X_train,
    y_train,
    base_model,
    param_grid,
    *,
    oversampling_mode=2,          # 1=SMOTE, 2=ADASYN
    sampling_ratio="not majority",
    rnn_k=5,
    scoring="f1_macro",
    n_splits=3,
    random_state=42,
    verbose=1
):
    """
    GridSearchCV over an imblearn pipeline:
        imputer -> scaler -> RNN(healthy-only) -> oversampler -> classifier
    """
    y_train = np.asarray(y_train)

    classes, counts = np.unique(y_train, return_counts=True)
    min_count = counts.min()

    n_splits = min(int(n_splits), int(min_count))
    if n_splits < 2:
        raise ValueError(
            f"n_splits became {n_splits}. Need at least 2 samples per class. "
            f"Class distribution: {dict(zip(classes, counts))}"
        )

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    overs_name = "SMOTE" if oversampling_mode == 1 else "ADASYN"
    oversampler = get_oversampler(
        name=overs_name,
        y=y_train,
        n_splits=n_splits,
        sampling_ratio=sampling_ratio,
        random_state=random_state
    )

    pipe = ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
        ("rnn_clean", make_rnn_sampler(k=rnn_k)),
        ("oversampler", oversampler),
        ("clf", clone(base_model)),
    ])

    grid_prefixed = {f"clf__{k}": v for k, v in param_grid.items()}

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=grid_prefixed,
        scoring=scoring,     # multiclass macro-F1
        cv=cv,
        n_jobs=-1,
        verbose=verbose,
        error_score="raise",
        refit=True
    )

    grid.fit(X_train, y_train)

    return {
        "best_estimator": grid.best_estimator_,
        "best_params": grid.best_params_,
        "best_score": float(grid.best_score_),
        "cv_results": grid.cv_results_,
        "n_splits": n_splits,
        "oversampler": overs_name,
        "sampling_ratio": sampling_ratio,
        "rnn_k": rnn_k,
        "scoring": scoring,
    }

In [ ]:
def summarize_grid_results(grid):
    results = pd.DataFrame(grid.cv_results_)

    cols_to_keep = [
        *[c for c in results.columns if c.startswith("param_")],

        "mean_train_f1_macro",
        "std_train_f1_macro",
        "mean_test_f1_macro",
        "std_test_f1_macro",

        # these must exist in scoring, otherwise remove them
        *[c for c in [
            "mean_train_accuracy",
            "mean_test_accuracy",
            "mean_train_recall_macro",
            "mean_test_recall_macro"
        ] if c in results.columns],

        "mean_fit_time",
        "mean_score_time",
    ]

    summary = results[cols_to_keep].copy()

    # Add overfitting gap column
    summary["f1_gap"] = (
        summary["mean_train_f1_macro"]
        - summary["mean_test_f1_macro"]
    )

    # Mark selected model BEFORE sorting
    summary["selected_by_refit"] = False
    summary.loc[grid.best_index_, "selected_by_refit"] = True

    # Sort by validation F1 descending
    summary = summary.sort_values(
        by="mean_test_f1_macro",
        ascending=False
    ).reset_index(drop=True)

    return summary


In [ ]:
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import make_scorer, f1_score
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

from sklearn.metrics import accuracy_score, recall_score

def _best_index_with_gap(cv_results, *, scorer="f1_macro", max_gap=0.05):
    """
    Select the candidate with highest mean_test_<scorer>
    among those with (mean_train_<scorer> - mean_test_<scorer>) <= max_gap.
    If none satisfy, fall back to the best mean_test_<scorer>.
    """
    test_key  = f"mean_test_{scorer}"
    train_key = f"mean_train_{scorer}"

    test_scores  = np.asarray(cv_results[test_key], dtype=float)
    train_scores = np.asarray(cv_results[train_key], dtype=float)

    gaps = train_scores - test_scores  # absolute gap in F1 units (0..1)

    ok = gaps <= max_gap
    if np.any(ok):
        # among acceptable candidates, pick the highest validation score
        best = np.argmax(np.where(ok, test_scores, -np.inf))
    else:
        # fallback: standard best by validation score
        best = int(np.argmax(test_scores))
    return int(best)

def tune_one_model(
    X_train,
    y_train,
    base_model,
    param_grid,
    *,
    oversampling_mode=2,
    sampling_ratio="not majority",
    rnn_k=5,
    n_splits=3,
    random_state=42,
    verbose=1,
    debug_logging=False,
    max_overfit_gap=0.05,   # ← NEW: 5% (i.e., 0.05 F1)
):
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)

    classes, counts = np.unique(y_train, return_counts=True)
    min_count = counts.min()
    n_splits = min(int(n_splits), int(min_count))
    if n_splits < 2:
        raise ValueError(
            f"n_splits became {n_splits}. Need at least 2 samples per class. "
            f"Class distribution: {dict(zip(classes, counts))}"
        )
    n_repeats = 5  # or 10 if you can afford runtime
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=random_state
    )
    # cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # Oversampler selection (your helper)
    if oversampling_mode == 1:
        overs_name = "SMOTE"
    elif oversampling_mode == 2:
        overs_name = "ADASYN"
    else:
        raise ValueError("oversampling_mode must be 1 (SMOTE) or 2 (ADASYN).")

    oversampler = get_oversampler(
        name=overs_name,
        y=y_train,
        n_splits=n_splits,
        sampling_ratio=sampling_ratio,
        random_state=random_state
    )

    pipeline_steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
        ("rnn_clean", make_rnn_sampler(k=rnn_k)),
    ]
    if debug_logging:
        pipeline_steps.append(("log_after_rnn", DataSizeLogger(step_name="After RNN")))

    pipeline_steps.append(("oversampler", oversampler))

    if debug_logging:
        pipeline_steps.append(("log_after_oversample", DataSizeLogger(step_name="After Oversampling")))

    pipeline_steps.append(("clf", clone(base_model)))

    pipe = ImbPipeline(pipeline_steps)

    grid_prefixed = {f"clf__{k}": v for k, v in param_grid.items()}

    # Multi-metric scoring (you can add more metrics later if you want)
    scoring = {
        "f1_macro": make_scorer(f1_score, average="macro"),
        "accuracy": make_scorer(accuracy_score),
        "recall_macro": make_scorer(recall_score, average="macro"),
    }

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=grid_prefixed,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        verbose=verbose,
        error_score="raise",
        return_train_score=True,  # ← REQUIRED to compute the overfit gap
        refit=lambda cv_results: _best_index_with_gap(
            cv_results, scorer="f1_macro", max_gap=max_overfit_gap
        ),
    )

    grid.fit(X_train, y_train)

    # compute the chosen candidate's gap for reporting
    best_i = grid.best_index_
    mean_train = float(grid.cv_results_["mean_train_f1_macro"][best_i])
    mean_test  = float(grid.cv_results_["mean_test_f1_macro"][best_i])
    chosen_gap = mean_train - mean_test

    return {
        "best_estimator": grid.best_estimator_,
        "best_params": grid.best_params_,
        "best_score": float(mean_test),
        "chosen_overfit_gap": float(chosen_gap),
        "cv_results": grid.cv_results_,
        "n_splits": n_splits,
        "oversampler": overs_name,
        "sampling_ratio": sampling_ratio,
        "rnn_k": rnn_k,
        "scoring": "f1_macro_with_gap_constraint",
        "max_overfit_gap": max_overfit_gap,
        "grid_object": grid,   # optional but useful
    }
    



In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold
import numpy as np
from collections import Counter

def sampling_dict_ratio_to_majority(y, majority_class=0, ratio=0.5):
    c = Counter(y)
    n0 = c[majority_class]
    target = int(np.floor(ratio * n0))
    return {cls: target for cls, n in c.items() if cls != majority_class and n < target}


def tune_with_oversampler_cv(
    X,
    y,
    model,
    param_grid,
    scoring="f1_macro",
    n_splits=3,
    n_repeats=1,              # <-- NEW: set to 1 for standard StratifiedKFold
    oversampler_name="SMOTE",
    sampling_ratio="not majority",
    random_state=42,
    verbose=1,
    n_jobs=1                  # <-- NEW: made configurable
):
    """
    Tune a model with an oversampler inside CV (no leakage).
    Supports both standard and repeated stratified k-fold.
    
    Parameters
    ----------
    X : array-like
        Features
    y : array-like
        Target labels
    model : estimator
        Base model to tune
    param_grid : dict
        Hyperparameter grid (without "clf__" prefix, will be added automatically)
    scoring : str or callable
        Scoring metric for GridSearchCV
    n_splits : int
        Number of CV folds
    n_repeats : int, default=1
        Number of times to repeat CV with different random seeds.
        Set to 1 for standard StratifiedKFold.
        Set to >1 (e.g., 3, 5, 10) for RepeatedStratifiedKFold.
    oversampler_name : str
        Name of oversampler (passed to get_oversampler)
    sampling_ratio : float or str
        Sampling strategy (passed to get_oversampler)
    random_state : int
        Random seed
    verbose : int
        Verbosity level for GridSearchCV
    n_jobs : int
        Number of parallel jobs (-1 for all cores)
    
    Returns
    -------
    best_estimator : Pipeline
        Best fitted pipeline
    best_params : dict
        Best hyperparameters (with "clf__" prefix)
    best_score : float
        Best cross-validation score
    cv_results : dict
        Full CV results from GridSearchCV
    
    Notes
    -----
    Uses your CV-safe get_oversampler signature:
        get_oversampler(name, y, n_splits, sampling_ratio, random_state)
    """

    X = np.asarray(X)
    y = np.asarray(y)

    # --- class counts + safe n_splits ---
    classes, counts = np.unique(y, return_counts=True)
    min_count = counts.min()

    n_splits_eff = min(int(n_splits), int(min_count))
    if n_splits_eff < 2:
        raise ValueError(
            f"n_splits became {n_splits_eff}. Need at least 2 samples per class. "
            f"Class distribution: {dict(zip(classes, counts))}"
        )

    print(f"Class distribution: {dict(zip(classes, counts))}")
    print(f"Using n_splits={n_splits_eff}, n_repeats={n_repeats}, "
          f"oversampler={oversampler_name}, sampling_ratio={sampling_ratio}")

    # --- Create CV strategy ---
    if n_repeats > 1:
        cv = RepeatedStratifiedKFold(
            n_splits=n_splits_eff,
            n_repeats=n_repeats,
            random_state=random_state
        )
        cv_type = "RepeatedStratifiedKFold"
        total_fits = n_splits_eff * n_repeats * len(list(_grid_size(param_grid)))
    else:
        cv = StratifiedKFold(
            n_splits=n_splits_eff,
            shuffle=True,
            random_state=random_state
        )
        cv_type = "StratifiedKFold"
        total_fits = n_splits_eff * len(list(_grid_size(param_grid)))

    print(f"CV strategy: {cv_type}")
    print(f"Total fits: ~{total_fits} (approx)")

    # --- create oversampler ---
    oversampler = get_oversampler(
        name=oversampler_name,
        y=y,
        n_splits=n_splits_eff,
        sampling_ratio=sampling_ratio,
        random_state=random_state
    )

    # --- build pipeline ---
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
        ("oversampler", oversampler),
        ("clf", model),
    ])

    # prefix grid with clf__
    param_grid_prefixed = {f"clf__{k}": v for k, v in param_grid.items()}

    # --- GridSearchCV ---
    grid = GridSearchCV(
        pipe,
        param_grid_prefixed,
        scoring=scoring,
        cv=cv,
        n_jobs=n_jobs,
        verbose=verbose,
        error_score="raise",
        return_train_score=False  # saves memory
    )

    grid.fit(X, y)

    # --- Extract best results ---
    best_params_clean = {
        k.replace("clf__", ""): v for k, v in grid.best_params_.items()
    }

    print(f"\nBest score ({scoring}): {grid.best_score_:.4f}")
    print(f"Best params: {best_params_clean}")

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


def _grid_size(param_grid):
    """Helper to estimate grid size."""
    import itertools
    if not param_grid:
        return 1
    return itertools.product(*param_grid.values())

## Model Selection

### Cross Validation

In [ ]:
# Cross Validation to compare model performances 
# SMOTE applied within CV folds

cv_summary = crossval_metrics(
    X_train_scaled, y_train_use,
    model_group="supervised_smote",
    use_smote_in_cv=True,
    oversampler_name="SMOTE",
    sampling_ratio="not majority",
    average="macro",
    n_splits=5,
)

cv_summary

In [ ]:
crossval_metrics_repeated(
    X_train_scaled, y_train_use,
    model_group="supervised_smote",
    use_smote_in_cv=True,
    oversampler_name="SMOTE",
    sampling_ratio=0.5,
    average="macro",
    n_splits=5,
    n_repeats=5,
)

From the Cross validaiton results we choose our desired model (best F1-score, or best Recall, or best ROC-AUC)
Lets say in this case KNN, RF, and SVM_rbf is good

# Hyperparameter Tuning based on f1_macro

In [ ]:
strategy = sampling_dict_ratio_to_majority(y, majority_class=0, ratio=0.5)

param_grid_rf={
        "n_estimators": [50, 100],
        "max_depth": [3,4],
        "min_samples_split": [2,3,4],
        "min_samples_leaf": [2,3,5],
    }
rf_results = tune_one_model(
    X_train=X_train,
    y_train=y_train,
    base_model=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    n_splits=5
)

# Access results
best_model_rf = rf_results['best_estimator']
best_params_rf = rf_results['best_params']
best_score_rf = rf_results['best_score']

print(best_params_rf)
print(best_score_rf)


In [ ]:
param_grid_svm = {
    "kernel": ["rbf"],
    "C": [0.001,0.01,0.1,1,10],       # [0.01, 0.1, 1, 10, 100]
    "gamma": ["scale",0.001,0.01,0.1],   # [0.001, 0.01, 0.1, 1, 10]
    "class_weight": [None],                    # optional; see note below
    "probability": [True],                     # only if you need predict_proba
}
svm_results = tune_one_model(
    X_train=X_train,
    y_train=y_train,
    base_model=SVC(kernel="rbf", probability=True),
    param_grid=param_grid_svm,
    n_splits=5
)

# Access results
best_model_svm = svm_results['best_estimator']
best_params_svm = svm_results['best_params']
best_score_svm = svm_results['best_score']

print(best_params_svm)
print(best_score_svm)
cv_svm = svm_results["cv_results"]
print([k for k in svm_results["cv_results"].keys() if "test" in k])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_cv_svm = pd.DataFrame(svm_results["cv_results"])

# Example: plot vs C for a fixed gamma
df_plot = df_cv_svm[df_cv_svm["param_clf__gamma"].astype(str) == "scale"].copy()

# Coerce to numeric safely
df_plot["C_num"] = pd.to_numeric(df_plot["param_clf__C"], errors="coerce")

# Drop invalid / non-positive C (required for log scale)
df_plot = df_plot.dropna(subset=["C_num"])
df_plot = df_plot[df_plot["C_num"] > 0].sort_values("C_num")

C_vals = df_plot["C_num"].values
train_mean = df_plot["mean_train_f1_macro"].values
train_std  = df_plot["std_train_f1_macro"].values
test_mean  = df_plot["mean_test_f1_macro"].values
test_std   = df_plot["std_test_f1_macro"].values

plt.figure(figsize=(8,5))
plt.plot(C_vals, train_mean, linewidth=2, label="Train F1 (macro)")
plt.fill_between(C_vals, train_mean-train_std, train_mean+train_std, alpha=0.2)

plt.plot(C_vals, test_mean, linewidth=2, label="Validation F1 (macro)")
plt.fill_between(C_vals, test_mean-test_std, test_mean+test_std, alpha=0.2)

plt.xscale("log")  # now safe
plt.xlabel("C (log scale)")
plt.ylabel("F1 Macro")
plt.title("SVM Tuning (gamma=scale)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
param_grid_knn={
        "n_neighbors": [7,9, 11, 13],
        "weights": ["uniform"],
    }
knn_results = tune_one_model(
    X_train=X_train,
    y_train=y_train,
    base_model=KNeighborsClassifier(),
    param_grid=param_grid_knn,
    n_splits=5
)

# Access results
best_model_knn = knn_results['best_estimator']
best_params_knn = knn_results['best_params']
best_score_knn = knn_results['best_score']

print(best_params_knn)
print(best_score_knn)
cv_knn = knn_results["cv_results"]
print([k for k in cv_knn.keys() if "train" in k])
print([k for k in cv_knn.keys() if "test" in k])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df_cv_knn = pd.DataFrame(knn_results["cv_results"])

df_plot = df_cv_knn[df_cv_knn["param_clf__weights"] == "uniform"]

k_vals = df_plot["param_clf__n_neighbors"].astype(int)

# F1 SCORE PLOT
train_mean = df_plot["mean_train_f1_macro"]
train_std  = df_plot["std_train_f1_macro"]

test_mean = df_plot["mean_test_f1_macro"]
test_std  = df_plot["std_test_f1_macro"]

plt.figure(figsize=(8,5))

# Training curve
plt.plot(k_vals, train_mean, linewidth=2, label="Train F1 (macro)")
plt.fill_between(k_vals,
                 train_mean - train_std,
                 train_mean + train_std,
                 alpha=0.2)

# Validation curve
plt.plot(k_vals, test_mean, linewidth=2, label="Validation F1 (macro)")
plt.fill_between(k_vals,
                 test_mean - test_std,
                 test_mean + test_std,
                 alpha=0.2)

plt.xlabel("Number of Neighbors (k)")
plt.ylabel("F1 Macro")
plt.title("KNN Hyperparameter Tuning (Repeated CV)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# RECALL PLOT
recall_train_mean = df_plot["mean_train_recall_macro"]
recall_train_std  = df_plot["std_train_recall_macro"]

recall_test_mean = df_plot["mean_test_recall_macro"]
recall_test_std  = df_plot["std_test_recall_macro"]

plt.figure(figsize=(8,5))

plt.plot(k_vals, recall_train_mean, linewidth=2, label="Train Recall (macro)")
plt.fill_between(k_vals,
                 recall_train_mean - recall_train_std,
                 recall_train_mean + recall_train_std,
                 alpha=0.2)

plt.plot(k_vals, recall_test_mean, linewidth=2, label="Validation Recall (macro)")
plt.fill_between(k_vals,
                 recall_test_mean - recall_test_std,
                 recall_test_mean + recall_test_std,
                 alpha=0.2)

plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Recall Macro")
plt.title("KNN Hyperparameter Tuning (Recall)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# best_model_dt, best_params_dt, best_score_dt, cvres_dt = tune_with_oversampler_cv(
#     X_train, y_train,
#     model=DecisionTreeClassifier(random_state=42),
#     param_grid={
#         "max_depth": [3, 4, 5],
#         "min_samples_split": [2, 5],
#         "min_samples_leaf": [1, 2],
#     },
#     scoring="f1_macro",
#     oversampler_name="ADASYN",
#     sampling_ratio=strategy,
#     n_splits=5
# )
# print("Best Decision Tree (with ADASYN) parameters and score:")
# print(best_params_dt)
# print(best_score_dt)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

def plot_tuning_results(rf_results, param_grid):
    """
    Visualize F1 scores across grid search iterations.
    
    Parameters:
    -----------
    rf_results : dict
        Results dictionary from tune_one_model()
    param_grid : dict
        Original parameter grid (without 'clf__' prefix)
    """
    cv_results = rf_results['cv_results']
    best_score = rf_results['best_score']
    
    # Extract mean test scores and parameters
    mean_scores = cv_results['mean_test_f1_macro']
    # std_scores = cv_results['std_test_score']
    params = cv_results['params']
    
    # Remove 'clf__' prefix from parameter names for cleaner display
    params_clean = [{k.replace('clf__', ''): v for k, v in p.items()} for p in params]
    
    # Create readable labels
    param_labels = []
    for p in params_clean:
        label_parts = [f"{k}={v}" for k, v in p.items()]
        param_labels.append('\n'.join(label_parts))
    
    # ========== Plot 1: Line plot with error bars ==========
    plt.figure(figsize=(16, 6))
    
    x_pos = np.arange(len(mean_scores))
    plt.errorbar(x_pos, mean_scores, yerr=None, 
                 marker='o', linestyle='-', linewidth=2, capsize=5, capthick=2)
    
    plt.xlabel('Parameter Combination', fontsize=12)
    plt.ylabel(f'{rf_results["scoring"]} Score', fontsize=12)
    plt.title(f'Grid Search Results - {rf_results["oversampler"]} (k={rf_results["rnn_k"]}, ratio={rf_results["sampling_ratio"]})', 
              fontsize=14, fontweight='bold')
    plt.xticks(x_pos, param_labels, rotation=45, ha='right', fontsize=9)
    plt.grid(True, alpha=0.3, axis='y')
    plt.axhline(y=best_score, color='r', linestyle='--', linewidth=2, 
                label=f'Best Score: {best_score:.4f}')
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
    
    # ========== Plot 2: Heatmap (if 2D grid) ==========
    # Only works if you have exactly 2 parameters
    if len(param_grid) == 2:
        param_names = list(param_grid.keys())
        param1_vals = param_grid[param_names[0]]
        param2_vals = param_grid[param_names[1]]
        
        # Reshape scores into 2D matrix
        scores_matrix = mean_scores.reshape(len(param1_vals), len(param2_vals))
        
        plt.figure(figsize=(10, 7))
        sns.heatmap(scores_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
                    xticklabels=param2_vals, 
                    yticklabels=param1_vals,
                    cbar_kws={'label': f'{rf_results["scoring"]} Score'},
                    vmin=mean_scores.min(), vmax=mean_scores.max(),
                    linewidths=0.5, linecolor='gray')
        plt.xlabel(param_names[1], fontsize=12, fontweight='bold')
        plt.ylabel(param_names[0], fontsize=12, fontweight='bold')
        plt.title(f'Hyperparameter Heatmap - {rf_results["oversampler"]}', 
                  fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    # ========== Print Top 5 Configurations ==========
    top_indices = np.argsort(mean_scores)[-5:][::-1]
    print("\n" + "="*60)
    print(f"TOP 5 CONFIGURATIONS ({rf_results['scoring']})")
    print("="*60)
    for i, idx in enumerate(top_indices, 1):
        params_str = ', '.join([f"{k}={v}" for k, v in params_clean[idx].items()])
        print(f"{i}. {params_str}")
        print(f"   Score: {mean_scores[idx]:.4f}")
        print()


# ========== USAGE ==========
# Visualize results
plot_tuning_results(knn_results, param_grid_knn)

# Grid Search Evaluation

In [ ]:
summary_svm = summarize_grid_results(svm_results["grid_object"])
summary_rf = summarize_grid_results(rf_results["grid_object"])
summary_knn = summarize_grid_results(knn_results["grid_object"])

def extract_selected_row(summary_df, model_name):
    row = summary_df[summary_df["selected_by_refit"]].copy()
    row.insert(0, "Model", model_name)
    return row

grid_output = pd.concat([
    summary_svm[summary_svm["selected_by_refit"]].assign(Model="SVM"),
    summary_rf[summary_rf["selected_by_refit"]].assign(Model="RandomForest"),
    summary_knn[summary_knn["selected_by_refit"]].assign(Model="KNN"),
], ignore_index=True)

# Move Model column to front
cols = ["Model"] + [c for c in grid_output.columns if c != "Model"]
grid_output = grid_output[cols]

# --- Step 2: Collapse param_ columns into single column ---
param_cols = [c for c in grid_output.columns if c.startswith("param_")]

grid_output["Parameters"] = (
    grid_output[param_cols]
    .apply(lambda row: ", ".join(
        f"{c.replace('param_clf__','').replace('param_','')}={row[c]}"
        for c in param_cols
        if pd.notna(row[c])
    ), axis=1)
)

# --- Step 3: Drop original param columns ---
grid_output = grid_output.drop(columns=param_cols)

display(grid_output)

grid_output.to_excel("grid_search_multi.xlsx", index=False)


### Model Evaluation on Test

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, f1_score, precision_score, recall_score
)

def evaluate_models_on_test(models_dict, X_test, y_test, average="macro"):
    """
    models_dict: {"RF": best_rf, "SVM": best_svm, "KNN": best_knn}
                where each item is a fitted estimator / pipeline
    """
    X_test = np.asarray(X_test)
    y_test = np.asarray(y_test)

    rows = []
    cms = {}

    for name, model in models_dict.items():
        y_pred = model.predict(X_test)

        rows.append({
            "Model": name,
            "BalancedAcc": balanced_accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, average=average, zero_division=0),
            "Recall": recall_score(y_test, y_pred, average=average, zero_division=0),
            "F1": f1_score(y_test, y_pred, average=average, zero_division=0)
        })

        cms[name] = confusion_matrix(y_test, y_pred)

        print("\n" + "="*70)
        print(f"{name} — classification report (test)")
        print("="*70)
        print(classification_report(y_test, y_pred, zero_division=0))

    df = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
    return df, cms


### Tuned Model Dictionary

In [ ]:
# Collect best models into a dict
models_dict = {
    "RF": best_model_rf,
    "SVM": best_model_svm,
    "KNN": best_model_knn,
    # "DT (tuned)": best_model_dt
}


In [ ]:
test_summary, test_confmats = evaluate_models_on_test(
    models_dict, X_test, y_test, average="macro"
)

test_summary


In [ ]:
plt.rcParams.update({
    'axes.titlesize': 22,
    'axes.labelsize': 20,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white'
})

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confmat(cm, title):
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["0", "1","2"],
        yticklabels=["0", "1","2"]
    )
    plt.title(title)
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

# Loop through your confusion matrices
for model_name, cm in test_confmats.items():
    plot_confmat(cm, f"Confusion Matrix — {model_name}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

def plot_confmats_side_by_side(
    confmats_dict,                 # {"rf_tuned": cm, "svm_tuned": cm, ...}
    class_labels=None,             # e.g. ["0","1","2"] or ["H","Aux","Comb"]
    max_cols=4,                    # wrap to new row if many models
    figsize_per_subplot=(4.2, 3.6),
    cmap="Blues",
    show_accuracy=True,
    shared_colorbar=True,
):
    model_names = list(confmats_dict.keys())
    if len(model_names) == 0:
        raise ValueError("confmats_dict is empty")

    # Infer labels from matrix size if not provided
    first_cm = confmats_dict[model_names[0]]
    n_classes = first_cm.shape[0]
    if class_labels is None:
        class_labels = [str(i) for i in range(n_classes)]
    else:
        if len(class_labels) != n_classes:
            raise ValueError(f"class_labels length ({len(class_labels)}) != n_classes ({n_classes})")

    # --- shared vmax for consistent color scaling
    vmax = max(int(np.max(cm)) for cm in confmats_dict.values())

    n_models = len(model_names)
    ncols = min(max_cols, n_models)
    nrows = math.ceil(n_models / ncols)

    fig_w = figsize_per_subplot[0] * ncols
    fig_h = figsize_per_subplot[1] * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), constrained_layout=True)

    axes = np.atleast_1d(axes).reshape(nrows, ncols)

    last_im = None  # for shared colorbar
    for idx, model_name in enumerate(model_names):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]
        cm = confmats_dict[model_name]

        # accuracy
        acc = np.trace(cm) / np.sum(cm) if np.sum(cm) > 0 else np.nan

        # seaborn heatmap on a given axis
        hm = sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap=cmap,
            vmin=0,
            vmax=vmax,
            annot_kws={"size": 18},
            xticklabels=class_labels,
            yticklabels=class_labels,
            cbar=False,          # we add shared colorbar manually below
            ax=ax
        )
        last_im = hm.collections[0]  # store mappable

        title = model_name
        if show_accuracy and np.isfinite(acc):
            title += f"\naccuracy = {acc*100:.1f}%"
        ax.set_title(title, fontsize=20, fontweight="bold")

        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")

    # turn off unused axes
    for j in range(n_models, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    # # shared colorbar (single, like publication figures)
    # if shared_colorbar and last_im is not None:
    #     cbar = fig.colorbar(last_im, ax=axes.ravel().tolist(), shrink=0.85, pad=0.02)
    #     cbar.set_label("Count")

    plt.show()


In [ ]:
plot_confmats_side_by_side(
    test_confmats,
    class_labels=["0", "1", "2"],   # or your class names
    max_cols=4
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def _surface_scores(model, X_grid, classes):
    """
    Returns:
      score: 1D continuous scalar for background coloring
             - Binary: signed decision_function if available else (proba - 0.5)
             - Multiclass: margin = top1 - top2 (from decision scores or probabilities)
      yhat : predicted labels for class boundary lines
    """
    yhat = model.predict(X_grid)

    # --- Prefer decision_function for SVC (gives signed distance in binary) ---
    if hasattr(model, "decision_function"):
        df = np.asarray(model.decision_function(X_grid))

        # Binary
        if df.ndim == 1:
            score = df  # signed distance (centered at 0)
            return score, yhat

        # Multiclass (OVR or OVO): build a comparable scalar "margin"
        # If OVR: df shape (n, n_classes) -> use top1-top2
        if df.ndim == 2:
            top2 = np.partition(df, -2, axis=1)[:, -2:]  # last two columns are top-2 (unordered)
            # ensure top1 >= top2
            top1 = np.max(top2, axis=1)
            top2v = np.min(top2, axis=1)
            score = top1 - top2v
            return score, yhat

    # --- Else use probabilities (works if probability=True) ---
    if hasattr(model, "predict_proba"):
        proba = np.asarray(model.predict_proba(X_grid))

        # Binary: center at 0 for a boundary at 0
        if proba.shape[1] == 2:
            pos_label = 1 if 1 in classes else classes[-1]
            pos_idx = np.where(classes == pos_label)[0][0]
            score = proba[:, pos_idx] - 0.5  # boundary at 0
            return score, yhat

        # Multiclass: margin top1-top2 in probability space
        top2 = np.partition(proba, -2, axis=1)[:, -2:]
        top1 = np.max(top2, axis=1)
        top2v = np.min(top2, axis=1)
        score = top1 - top2v
        return score, yhat

    return None, yhat



def plot_decision_surfaces_train_test(
    models_dict,
    X_train,
    y_train,
    X_test,
    y_test,
    use_pca=True,
    pca_components=2,
    grid_step=0.05,
    title_prefix="Decision Surface",
):
    """
    Produces one figure per model:
      left = train surface + points
      right = test surface + points

    Works for binary and multiclass. For binary, also draws a boundary contour.
    """

    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)
    X_test  = np.asarray(X_test)
    y_test  = np.asarray(y_test)

    classes = np.unique(np.concatenate([y_train, y_test]))
    is_binary = (len(classes) == 2)

    # --- 1) 2D embedding ---
    if use_pca:
        pca = PCA(n_components=pca_components, random_state=42)
        Xtr_2d = pca.fit_transform(X_train)   # fit ONLY on train
        Xte_2d = pca.transform(X_test)
        x_label, y_label = "PC1", "PC2"
    else:
        # assumes X has exactly 2 columns already
        if X_train.shape[1] != 2:
            raise ValueError("use_pca=False requires X_train to have exactly 2 features.")
        Xtr_2d, Xte_2d = X_train, X_test
        x_label, y_label = "TPE", "SDD"

    # --- 2) common grid limits (use TRAIN limits and reuse for TEST to match your example) ---
    x_min, x_max = Xtr_2d[:, 0].min() - 1, Xtr_2d[:, 0].max() + 1
    y_min, y_max = Xtr_2d[:, 1].min() - 1, Xtr_2d[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, grid_step),
        np.arange(y_min, y_max, grid_step),
    )
    X_grid_2d = np.c_[xx.ravel(), yy.ravel()]

    # --- 3) plot per model ---
    for name, model in models_dict.items():

        # We must evaluate the model in the same feature space it was trained on.
        # If you used PCA only for visualization, we need a "visual-space model".
        # Easiest: train a *copy* of the model on the 2D PCA features for plotting only.
        #
        # This does NOT replace your real model; it's just to draw the surface in 2D.
        # (Exactly what your screenshot implies: surface in PC1-PC2 space.)
        from sklearn.base import clone
        vis_model = clone(model)
        vis_model.fit(Xtr_2d, y_train)

        # scores and predicted regions on the grid
        score, yhat_grid = _surface_scores(vis_model, X_grid_2d, classes)
        Z_cls = yhat_grid.reshape(xx.shape)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
        fig.suptitle(f"{title_prefix} — {name}", fontsize=22)

        # Force white background (figure + axes)
        fig.patch.set_facecolor("white")
        for ax in axes.ravel():
            ax.set_facecolor("white")


        # --- helper to draw one panel ---
        def _draw_panel(ax, X2d, y, panel_title):
            from matplotlib.colors import ListedColormap
            
            # Define consistent colors for class regions
            class_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
                            '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
            n_classes = len(classes)
            cmap_classes = ListedColormap(class_colors[:n_classes])
            
            # Keep both layers but make score layer grayscale
            ax.contourf(xx, yy, Z_cls, alpha=0.5, cmap=cmap_classes, 
                        levels=np.arange(-0.5, n_classes, 1))

            if score is not None:
                Zs = score.reshape(xx.shape)
                ax.contourf(xx, yy, Zs, alpha=0.15, cmap='gray')  # Low alpha, grayscale

                # # 3) boundary line (binary only)
                # if is_binary:
                #     # For binary: if using decision_function, boundary is 0
                #     # If using prob, boundary is 0.5
                #     level = 0.0 if (hasattr(vis_model, "decision_function") and not hasattr(vis_model, "predict_proba")) else 0.5
                #     ax.contour(xx, yy, Zs, levels=[level], linewidths=2)

            # 4) scatter points
            for c in classes:
                mask = (y == c)
                ax.scatter(
                    X2d[mask, 0], X2d[mask, 1],
                    s=35, edgecolors="black", linewidths=0.5,
                    label=f"class {c}"
                )

            ax.set_title(panel_title)
            ax.set_xlabel(x_label)
            ax.set_ylabel(y_label)
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min, y_max)
            ax.grid(True, alpha=0.3)

        _draw_panel(axes[0], Xtr_2d, y_train, "Training")
        _draw_panel(axes[1], Xte_2d, y_test,  "Test")

        axes[0].legend(loc="best")
        plt.tight_layout()
        plt.show()


In [ ]:
plot_decision_surfaces_train_test(
    models_dict,
    X_train_scaled, y_train_use,
    X_test_scaled,  y_test,
    use_pca=False,
    grid_step=0.05,
    title_prefix="Decision Function"
)


# Save the model

In [ ]:
import json
import joblib
from pathlib import Path
from datetime import datetime

from sklearn.pipeline import Pipeline as SkPipeline

def sanitize_name(s: str) -> str:
    return str(s).lower().replace(" ", "_").replace("(", "").replace(")", "")

def make_run_id() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def ensure_dir(p: str | Path) -> Path:
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_inference_pipeline(fitted_imb_pipe) -> SkPipeline:
    """
    Keep only preprocessing + classifier.
    Drops samplers/cleaners (oversampler, rnn_clean) automatically.
    """
    steps = fitted_imb_pipe.named_steps
    return SkPipeline([
        ("imputer", steps["imputer"]),
        ("scaler",  steps["scaler"]),
        ("clf",     steps["clf"]),
    ])


In [ ]:
def save_model_artifacts(
    tuned_models: dict,
    save_dir: str | Path,
    run_id: str,
    experiment_tag: str,
    features: list[str],
    meta_base: dict,
):
    """
    For each model saves:
      - *_pipeline.joblib  : full fitted imblearn pipeline (includes samplers/cleaning)
      - *_inference.joblib : inference-only sklearn Pipeline (imputer+scaler+clf)
      - *_meta.json        : metadata for reproducibility
      - *_clf.joblib       : optional bundle containing classifier only (+features/meta)

    Notes:
    - For deployment/inference you almost always want *_inference.joblib
      because oversamplers are training-time only.
    """
    save_path = ensure_dir(save_dir)

    for model_name, fitted_pipe in tuned_models.items():
        base = sanitize_name(model_name)
        prefix = f"{run_id}_{experiment_tag}_"

        pipe_file  = save_path / f"{prefix}{base}_pipeline.joblib"
        infer_file = save_path / f"{prefix}{base}_inference.joblib"
        meta_file  = save_path / f"{prefix}{base}_meta.json"
        clf_file   = save_path / f"{prefix}{base}_clf.joblib"

        meta = dict(meta_base)
        meta.update({
            "model_name": model_name,
            "saved_at": datetime.now().isoformat(timespec="seconds"),
            "features": features,
            "artifacts": {
                "full_pipeline": pipe_file.name,
                "inference_pipeline": infer_file.name,
                "clf_bundle": clf_file.name,
                "meta_json": meta_file.name,
            }
        })

        # 1) full fitted pipeline (includes RNN cleaning + oversampling)
        joblib.dump(
            {"pipeline": fitted_pipe, "features": features, "meta": meta},
            pipe_file
        )

        # 2) inference-only pipeline (recommended for prediction usage)
        infer_pipe = make_inference_pipeline(fitted_pipe)
        joblib.dump(
            {"pipeline": infer_pipe, "features": features, "meta": meta},
            infer_file
        )

        # 3) optional classifier-only bundle (if you ever want just the estimator)
        clf = fitted_pipe.named_steps.get("clf", None)
        joblib.dump(
            {"model": clf, "features": features, "meta": meta},
            clf_file
        )

        # 4) readable metadata
        with open(meta_file, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

        print(f"[SAVED] {model_name}")
        print(f"  - full pipeline:     {pipe_file}")
        print(f"  - inference pipeline:{infer_file}")
        print(f"  - clf bundle:        {clf_file}")
        print(f"  - meta json:         {meta_file}\n")


In [ ]:
from pathlib import Path

# Current notebook working directory
NOTEBOOK_DIR = Path.cwd()

# Desired save path
SAVE_DIR = NOTEBOOK_DIR / "ML model multiclass" / "model"

print("Saving to:", SAVE_DIR)


RUN_ID = make_run_id()
EXPERIMENT_TAG = "feat2"
tuned_models = models_dict  # your dict of fitted pipelines (after tuning)
selected_features = selected_features  # list of features used for training
tuning_info = {
    "RF (tuned)": {
        "best_params": best_params_rf,
        "best_score_mean": float(best_score_rf),
    },
    "SVM (tuned)": {
        "best_params": best_params_svm,
        "best_score_mean": float(best_score_svm),
    },
    "KNN (tuned)": {
        "best_params": best_params_knn,
        "best_score_mean": float(best_score_knn),
    },
}


meta_base = {
    "run_id": RUN_ID,
    "experiment_tag": EXPERIMENT_TAG,
    "wv_bin": 1,
    "random_state": 42,
    # add your training config here
    "tuning_info": tuning_info,   # dict of best params per model, etc.
}

save_model_artifacts(
    tuned_models=tuned_models,
    save_dir=SAVE_DIR,
    run_id=RUN_ID,
    experiment_tag=EXPERIMENT_TAG,
    features=selected_features,
    meta_base=meta_base,
)
